[Markdown Cell]

# Phase 3 / 08 Refit Validation Winner

Use T4 x2. Attach notebook 00 prepared output, complete own Phase 1 A output and the matching tokenizer/metadata.
For refit also attach completed held-out Phase 3 task(s) and the winner family foundation/tokenizer; no Phase 2 task is used.


This notebook is independent and embeds its runtime. All train sentences and long tails are preserved. Model/tokenizer/checkpoint identities are verified, and actual shape/gradient checks precede two-rank training. A paused job retains exact state; it does not claim a completed submission. New Phase 3 F1 is unmeasured until execution.

The required five adaptations plus KD use all four architectures. This notebook's method is lorma. The official development splits select checkpoints. Private test labels are never model inputs.

[Markdown Cell]

Use this optional refit only if the completed **01 A LoRMA** is your chosen winner. Its original preparation helper differs from the later notebooks. This copy embeds the original exact helper so runtime checks remain intact. Attach prepared data, the complete completed 01 task, own Phase 1 A weights and its tokenizer. A report ZIP alone is insufficient. The refit trains on train+dev with a previously frozen epoch recipe and does not have a held-out F1.

[Markdown Cell]

## 1. Environment

In [5]:
# [Code Cell]
import importlib.metadata as package_metadata
import json
import os
from pathlib import Path
import subprocess
import sys

requirements = {'tokenizers': 'tokenizers==0.23.2', 'numpy': 'numpy==2.1.3',
                'scikit-learn': 'scikit-learn==1.7.2', 'tqdm': 'tqdm>=4.66,<5',
                'huggingface-hub': 'huggingface_hub>=0.30,<2'}
install = []
for package, requirement in requirements.items():
    try:
        version = package_metadata.version(package)
        if '==' in requirement and version != requirement.split('==')[1]: install.append(requirement)
    except package_metadata.PackageNotFoundError: install.append(requirement)
if install: subprocess.check_call([sys.executable,'-m','pip','install','-q',*install])
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Small dependencies ready. Keep Kaggle PyTorch unchanged; exact recovery checks its saved build.')


Small dependencies ready. Keep Kaggle PyTorch unchanged; exact recovery checks its saved build.


[Markdown Cell]

## 2. Paths, training budget and recovery

In [6]:
# [Code Cell]
EXPERIMENT_ID='P3_08_A01_Original_Runtime_Refit'
OUTPUT_ROOT='/kaggle/working/cs781_phase3/'+EXPERIMENT_ID
INPUTS=dict(prepared_dir=None,parent_run_dir=None,tokenizer_path=None,metadata_path=None,
            resume_run_dir=None,teacher_run_dir=None)
SOURCE_TASK_DIR=None # 08 only: exact completed held-out task folder containing job.json, or auto best.
MAX_EPOCHS=15
TRAINING_BUDGET_MINUTES=45 # Training only; preparation/dev/train/test inference adds overhead.
TARGET_TOTAL_EPOCHS=None # Completed task: set a higher total with identical settings to extend.
EXPECTED_PARENT_SHA256='0746fa1981f804a246b65d82f1d13e35c3a6bdf554bfff69df26a35dd1d5d131'
# Set expected hash to None only to accept another fully verified OWN completed continuation.
Path(OUTPUT_ROOT).mkdir(parents=True,exist_ok=True)


[Markdown Cell]

### Check actual mounted input paths

A Phase 1 root contains train_config.json, source and real checkpoint/receipts. A Phase 3 task root contains job.json, real task weights, receipt and exported logits. An .ipynb or prediction ZIP alone cannot provide weights.

In [7]:
# [Code Cell]
for pattern in ('train_config.json','prepared_manifest.json','phase3_prepared.zip','job.json','tokenizer.json'):
    for path in sorted(Path('/kaggle/input').rglob(pattern)): print(pattern,':',path)


train_config.json : /kaggle/input/notebooks/omprakashnetam69/modela-training-final/model_a_extended_run/train_config.json
prepared_manifest.json : /kaggle/input/notebooks/shubhamjoshi1806/00-download-audit-official-rr-data/cs781_phase3_data/prepared/prepared_manifest.json
phase3_prepared.zip : /kaggle/input/notebooks/shubhamjoshi1806/00-download-audit-official-rr-data/cs781_phase3_data/phase3_prepared.zip
job.json : /kaggle/input/notebooks/shubhamjoshi1806/01-a-lorma/cs781_phase3/P3_01_A_LoRMA/job.json
tokenizer.json : /kaggle/input/datasets/omprakashnetam69/dataset/kaggle/working/downloaded_output/downloaded_output/tokenizer.json
tokenizer.json : /kaggle/input/datasets/omprakashnetam69/dataset/kaggle/working/downloaded_output/tokenizer.json
tokenizer.json : /kaggle/input/notebooks/shubhamjoshi1806/01-a-lorma/cs781_phase3/P3_01_A_LoRMA/tokenizer/tokenizer.json


[Markdown Cell]

## 3. Model and measurement

Each complete sentence is split into bounded BPE pieces without truncating its tail. Mean and last-token foundation features are pooled per piece, projected and aggregated to one sentence vector. A packed bidirectional sentence GRU learns nearby document context; domain/position features are text/domain metadata. Gold roles appear only in the loss. Sentence order/coverage is asserted.

\[L_{CE}=\frac{\sum_i w_{y_i}\,CE(z_i,y_i)}{\sum_i w_{y_i}},\quad
L_{KD}=(1-\lambda)L_{CE}+\lambda T^2 KL(p_t^T\Vert p_s^T).\]

Class weights use training labels only. Development Macro-F1 averages all thirteen official labels, with combined CL/IT and per-domain/class diagnostics. A larger dense head alone is not a guaranteed improvement. Frozen train+dev refit is optional, fixed by an earlier best epoch and has no held-out development score.

References: [LoRMA](https://aclanthology.org/2025.findings-acl.527/), [IA3](https://huggingface.co/docs/peft/main/package_reference/ia3), [rsLoRA scaling](https://github.com/huggingface/peft/blob/main/docs/source/package_reference/lora.md). Implementations below use only own foundation weights.

In [8]:
# [Code Cell]
module_parts={}
def append_module(name,source):
    module_parts.setdefault(name,[]).append(source)


[Markdown Cell]

### __init__.py

Typed configuration, strict serialization and explicit task identity.

[Markdown Cell]

Importable source part 1

In [9]:
# [Code Cell]
append_module('__init__.py', r'''"""Own-foundation sentence-level rhetorical-role experiments."""
VERSION = 'cs781-rr-v1'
''')


[Markdown Cell]

### adaptation.py

LoRA: $W_0+\alpha BA/r$; rsLoRA changes scale to $\alpha/\sqrt r$. LoRMA+ uses $(I+\alpha BA/r)W_0$. IA3 learns K/V and feed-forward activation scales. Soft prompts learn input embeddings. Frozen weights remain the original own foundation.

[Markdown Cell]

Importable source part 1

In [10]:
# [Code Cell]
append_module('adaptation.py', r'''# CELL: Three low-rank methods with their actual distinct scaling/weight updates
import math
import torch
from torch import nn
from torch.nn import functional as F
from .config import RRConfig


class LowRank(nn.Module):
    """x[...,in] -> [...,out]; LoRA/rank-stabilized LoRA/LoRMA+ around frozen W0."""
    def __init__(self, base: nn.Linear, rank: int, alpha: float, method: str) -> None:
        super().__init__(); self.base = base; base.requires_grad_(False)
        self.method = method
        self.scale = alpha / (math.sqrt(rank) if method == 'rslora' else rank)
        self.a = nn.Parameter(torch.empty(rank, base.out_features if method == 'lorma' else base.in_features))
        self.b = nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.a, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = F.linear(x,self.base.weight,None)
        delta = F.linear(F.linear(y if self.method == 'lorma' else x,self.a),self.b)
        y = y + self.scale * delta
        return y if self.base.bias is None else y + self.base.bias


class ActivationScale(nn.Module):
    """IA3 scales K/V outputs or the FFN intermediate immediately before down projection."""
    def __init__(self, base: nn.Linear, mode: str) -> None:
        super().__init__(); self.base = base; base.requires_grad_(False); self.mode = mode
        width = base.in_features if mode == 'feedforward' else base.out_features // 3
        self.scales = nn.Parameter(torch.ones((2 if mode == 'qkv' else 1), width))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.mode == 'feedforward':
            return self.base(x * self.scales[0])
        query,key,value = self.base(x).chunk(3,dim=-1)
        return torch.cat((query,key*self.scales[0],value*self.scales[1]),dim=-1)


class PromptEmbedding(nn.Module):
    """Replace P reserved prefix embeddings with learned soft prompts; never add text labels."""
    def __init__(self, base: nn.Embedding, count: int) -> None:
        super().__init__(); self.base = base; base.requires_grad_(False)
        self.prompt = nn.Parameter(torch.empty(count,base.embedding_dim)); nn.init.normal_(self.prompt,std=0.02)

    @property
    def weight(self):
        return self.base.weight
''')


[Markdown Cell]

Importable source part 2

In [11]:
# [Code Cell]
append_module('adaptation.py', r'''
    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        hidden = self.base(ids); count = self.prompt.shape[0]
        return torch.cat((self.prompt[None].expand(ids.shape[0],-1,-1),hidden[:,count:]),dim=1)


def adapt(backbone: nn.Module, cfg: RRConfig) -> list[str]:
    """Adapt only active foundation modules; unused decoder/LM output adapters are excluded."""
    backbone.requires_grad_(False); changed = []
    if cfg.method == 'kd':
        backbone.requires_grad_(True)
        for module in (backbone.decoder,backbone.decoder_norm,backbone.target_positions):
            module.requires_grad_(False)
        return ['all active encoder/shared embedding weights']
    if cfg.method == 'prompt':
        backbone.token_embedding = PromptEmbedding(backbone.token_embedding,cfg.prompt_tokens)
        return ['token_embedding.prompt']
    prefix = {'a': 'layers.', 'b': 'encoder.', 'c': 'stages.'}[cfg.family]
    targets = [(name,module) for name,module in backbone.named_modules()
               if name.startswith(prefix) and isinstance(module,nn.Linear)]
    for name,module in targets:
        replacement = None
        if cfg.method in ('lora','rslora','lorma'):
            replacement = LowRank(module,cfg.rank,cfg.alpha,cfg.method)
        elif cfg.method == 'ia3':
            if name.endswith('attention.qkv'): replacement = ActivationScale(module,'qkv')
            elif name.endswith('ffn.down'): replacement = ActivationScale(module,'feedforward')
        if replacement is not None:
            parent_name,_,child = name.rpartition('.')
            setattr(backbone.get_submodule(parent_name),child,replacement); changed.append(name)
    if not changed:
        raise ValueError('No intended active foundation modules were adapted')
    return changed
''')


[Markdown Cell]

### config.py

Typed configuration, strict serialization and explicit task identity.

[Markdown Cell]

Importable source part 1

In [12]:
# [Code Cell]
append_module('config.py', r'''# CELL: Typed, identity-preserving experiment configuration
from dataclasses import asdict, dataclass


@dataclass(frozen=True)
class RRConfig:
    """Paths are portable; learning/head/data choices are immutable on recovery."""
    experiment_id: str = 'P3_E01_A_LoRMA'
    family: str = 'a'
    method: str = 'lorma'
    input_root: str = '/kaggle/input'
    run_dir: str = '/kaggle/working/cs781_phase3/P3_E01_A_LoRMA'
    prepared_dir: str | None = None
    parent_run_dir: str | None = None
    tokenizer_path: str | None = None
    metadata_path: str | None = None
    resume_run_dir: str | None = None
    teacher_run_dir: str | None = None
    required_teacher_sha256: str | None = None
    parent_checkpoint: str = 'best'
    expected_parent_sha256: str | None = None
    max_epochs: int = 15
    minimum_epochs: int = 4
    patience: int = 4
    target_total_epochs: int | None = None
    seed: int = 781
    world_size: int = 2
    micro_batch_size: int = 1
    accumulation_steps: int = 4
    num_workers: int = 0
    max_run_seconds: int = 45 * 60
    min_free_bytes: int = 1 << 30
    checkpoint_every_steps: int = 500
    max_consecutive_overflows: int = 8
    backbone_lr: float = 2e-4
    head_lr: float = 5e-4
    weight_decay: float = 0.01
    warmup_fraction: float = 0.1
    min_lr_ratio: float = 0.1
    max_grad_norm: float = 1.0
    rank: int = 16
    alpha: float = 32.0
    prompt_tokens: int = 16
    sentence_tokens: int = 192
    owned_sentences: int = 8
    context_sentences: int = 2
    piece_batch_size: int = 4
    head_width: int = 256
''')


[Markdown Cell]

Importable source part 2

In [13]:
# [Code Cell]
append_module('config.py', r'''    dropout: float = 0.2
    document_context: bool = True
    checkpoint_encoder: bool = True
    activation_checkpointing: bool = False
    class_weight_max: float = 4.0
    distillation_temperature: float = 2.0
    distillation_weight: float = 0.3
    selection_mode: str = 'heldout'
    fixed_epochs: int = 0
    refit_schedule_epochs: int = 15

    def validate(self):
        matrix = {'lorma': 'a', 'rslora': 'a', 'lora': 'b', 'ia3': 'c', 'prompt': 'd', 'kd': 'b'}
        if matrix.get(self.method) != self.family:
            raise ValueError('Use the declared five-method/four-foundation matrix plus B KD')
        if not self.experiment_id or any(c in self.experiment_id for c in '/\\'):
            raise ValueError('Use a simple experiment_id without path separators')
        if self.world_size != 2 or min(self.micro_batch_size, self.accumulation_steps) < 1:
            raise ValueError('Two GPUs and positive micro/accumulation batch sizes are required')
        if self.parent_checkpoint not in ('best', 'latest') or self.selection_mode not in ('heldout', 'refit'):
            raise ValueError('Invalid parent/selection policy')
        if self.selection_mode == 'refit' and (self.fixed_epochs < 1 or self.target_total_epochs):
            raise ValueError('Refit needs a fixed epoch recipe and cannot be extended using dev feedback')
        if min(self.max_epochs, self.minimum_epochs, self.patience) < 1:
            raise ValueError('Epoch/patience values must be positive')
        if self.target_total_epochs is not None and self.target_total_epochs < 1:
            raise ValueError('target_total_epochs must be positive')
        if not 8 <= self.sentence_tokens <= 512 or not 1 <= self.owned_sentences <= 32:
            raise ValueError('Invalid sentence token/chunk dimensions')
        if not 0 <= self.context_sentences <= 8 or not 1 <= self.piece_batch_size <= 16:
            raise ValueError('Invalid context/piece batch size')
        if self.head_width % 2 or not 32 <= self.head_width <= 512 or not 0 <= self.dropout < 1:
            raise ValueError('Even head width and valid dropout required')
        if not 1 <= self.rank <= 128 or not 0 < self.alpha <= 256 or not 1 <= self.prompt_tokens <= 64:
            raise ValueError('Invalid adapter/prompt dimensions')
        if not 0 < self.backbone_lr <= 1e-3 or not 0 < self.head_lr <= 1e-2:
            raise ValueError('Invalid learning rates')
        if not 1 <= self.class_weight_max <= 20 or not 0 < self.max_grad_norm <= 10:
            raise ValueError('Invalid weighting/clipping')
        if not 0 < self.distillation_weight < 1 or not 1 <= self.distillation_temperature <= 10:
            raise ValueError('Invalid KD settings')
        if self.checkpoint_every_steps != 500 or self.num_workers != 0:
            raise ValueError('Use 500-update recovery and deterministic zero-worker ragged batches')
        if self.max_run_seconds < 1 or not 0 <= self.warmup_fraction < 1 or not 0 < self.min_lr_ratio <= 1:
            raise ValueError('Invalid runtime/schedule')

    def choices(self):
        value = asdict(self)
''')


[Markdown Cell]

Importable source part 3

In [14]:
# [Code Cell]
append_module('config.py', r'''        for key in ('input_root','run_dir','prepared_dir','parent_run_dir','tokenizer_path','metadata_path',
                    'resume_run_dir','teacher_run_dir','max_run_seconds','min_free_bytes','max_epochs',
                    'target_total_epochs','parent_checkpoint','expected_parent_sha256'):
            value.pop(key)
        return value
''')


[Markdown Cell]

### data.py

Tokenize each complete sentence. Store ragged tokens and offsets; split long sentences into all bounded pieces. Window ownership supervises each original sentence once; neighboring sentences contribute context only. No training document or sentence is sampled away.

[Markdown Cell]

Importable source part 1

In [15]:
# [Code Cell]
append_module('data.py', r'''# CELL: Lossless ragged token caching with one owned target per original sentence
from array import array
import math
from pathlib import Path
import sys
import numpy as np
import torch
from tokenizers import Tokenizer
from tqdm.auto import tqdm
from .io_utils import atomic_json, identity, read_json, sha256_file
from .config import RRConfig


def cache_data(prepared: Path, tokenizer_path: Path, cfg: RRConfig) -> dict:
    """Tokenize every sentence once; long sentences are segmented, never truncated."""
    manifest = read_json(Path(prepared) / 'prepared_manifest.json')
    choices = {'prepared': manifest['identity'], 'tokenizer': sha256_file(tokenizer_path),
               'tokenizers': __import__('tokenizers').__version__, 'sentence_tokens': cfg.sentence_tokens,
               'owned': cfg.owned_sentences, 'context': cfg.context_sentences if cfg.document_context else 0,
               'selection_mode': cfg.selection_mode, 'weight_cap': cfg.class_weight_max}
    digest = identity(choices); root = Path(cfg.run_dir) / 'cache' / digest[:16]
    root.mkdir(parents=True, exist_ok=True)
    old = root / 'cache_manifest.json'
    if old.exists():
        cached = read_json(old)
        if cached['identity'] != digest:
            raise ValueError('Token cache identity changed')
        for name, receipt in cached['artifacts'].items():
            if sha256_file(root / name) != receipt:
                raise ValueError('Corrupt token cache: ' + name)
        return {**cached, 'directory': str(root)}
    tokenizer = Tokenizer.from_file(str(tokenizer_path)); labels = manifest['labels']
    label_to_id = {name: index for index, name in enumerate(labels)}
    tokens = array('H'); offsets = [0]; targets = []; docs = {}; windows = {}; train_counts = np.zeros(13, dtype=np.int64)
    rows_by_split = {s: read_json(Path(prepared) / (s + '.json')) for s in ('train','dev','test')}
    if cfg.selection_mode == 'refit':
        rows_by_split['train'] = [*rows_by_split['train'], *rows_by_split['dev']]
    if sys.byteorder != 'little':
        raise RuntimeError('This token cache targets little-endian Kaggle/Linux hosts')
    for split, rows in rows_by_split.items():
        docs[split] = []; windows[split] = []
        for row in tqdm(rows, desc='Tokenize ' + split):
            start = len(targets); length = len(row['text'])
            doc = {'id': row['id'], 'domain': row['domain'], 'start': start, 'sentences': length}
            doc_index = len(docs[split]); docs[split].append(doc)
            for index, text in enumerate(row['text']):
                encoded = tokenizer.encode(text, add_special_tokens=False).ids
                if any(token < 0 or token >= 32768 for token in encoded):
''')


[Markdown Cell]

Importable source part 2

In [16]:
# [Code Cell]
append_module('data.py', r'''                    raise ValueError('Unexpected token vocabulary')
                tokens.extend(encoded); offsets.append(len(tokens))
                target = label_to_id[row['labels'][index]] if split != 'test' else -100
                targets.append(target)
                if split == 'train': train_counts[target] += 1
            for owned_start in range(0, length, cfg.owned_sentences):
                owned_end = min(length, owned_start + cfg.owned_sentences)
                context = cfg.context_sentences if cfg.document_context else 0
                lo, hi = max(0, owned_start-context), min(length, owned_end+context)
                windows[split].append({'doc': doc_index, 'lo': lo, 'hi': hi,
                                       'owned_start': owned_start, 'owned_end': owned_end})
    if not windows['train']:
        raise ValueError('Training has no sentence targets')
    with (root / 'tokens.bin').open('wb') as stream:
        tokens.tofile(stream)
    np.save(root / 'offsets.npy', np.asarray(offsets, dtype=np.int64))
    np.save(root / 'targets.npy', np.asarray(targets, dtype=np.int64))
    positive = train_counts[train_counts > 0]
    weights = np.sqrt(max(1.0, float(positive.mean())) / np.maximum(train_counts, 1))
    weights = np.clip(weights, 0.25, cfg.class_weight_max)
    metadata = {'identity': digest, 'choices': choices, 'labels': labels, 'documents': docs, 'windows': windows,
                'class_weights': weights.tolist(), 'train_counts': train_counts.tolist(), 'sentences_dropped': 0,
                'token_count': len(tokens), 'sentence_count': len(targets), 'all_sentence_tails_preserved': True}
    metadata['artifacts'] = {name: sha256_file(root / name) for name in ('tokens.bin','offsets.npy','targets.npy')}
    atomic_json(old, metadata)
    return {**metadata, 'directory': str(root)}


class SentenceWindows:
    """Ragged examples carry text-derived IDs/positions and separate loss-only labels."""
    def __init__(self, cache: dict, split: str, cfg: RRConfig, teacher: np.ndarray | None = None) -> None:
        self.cache, self.split, self.cfg = cache, split, cfg
        root = Path(cache['directory'])
        self.tokens = np.memmap(root / 'tokens.bin', dtype='<u2', mode='r') if cache['token_count'] else np.empty(0, dtype=np.uint16)
        self.offsets = np.load(root / 'offsets.npy', mmap_mode='r')
        self.targets = np.load(root / 'targets.npy', mmap_mode='r')
        self.teacher = teacher

    def __len__(self) -> int:
        return len(self.cache['windows'][self.split])

    def __getitem__(self, index: int) -> dict:
        if index < 0:
            return {'pieces': [[1,2]], 'piece_sentence': [0], 'piece_weight': [1.0], 'positions': [[0.,0.]],
                    'domain': 0, 'targets': [-100], 'global_sentence': [-1], 'teacher': np.zeros((1,13), dtype=np.float32)}
        window = self.cache['windows'][self.split][index]
        doc = self.cache['documents'][self.split][window['doc']]
        pieces = []; mapping = []; piece_weights = []; targets = []; positions = []; global_ids = []; teacher = []
''')


[Markdown Cell]

Importable source part 3

In [17]:
# [Code Cell]
append_module('data.py', r'''        capacity = self.cfg.sentence_tokens - 2
        for local, sentence in enumerate(range(window['lo'], window['hi'])):
            global_id = doc['start'] + sentence
            begin, end = map(int, self.offsets[global_id:global_id+2])
            ids = self.tokens[begin:end]
            for first in range(0, max(1, len(ids)), capacity):
                content = ids[first:first+capacity].astype(np.int64).tolist()
                pieces.append([1, *content, 2]); mapping.append(local); piece_weights.append(float(max(1,len(content))))
            owned = window['owned_start'] <= sentence < window['owned_end']
            targets.append(int(self.targets[global_id]) if owned else -100)
            positions.append([sentence / max(1,doc['sentences']-1), math.log1p(doc['sentences']) / 10])
            global_ids.append(global_id if owned else -1)
            teacher.append(self.teacher[global_id] if self.teacher is not None else np.zeros(13, dtype=np.float32))
        return {'pieces': pieces, 'piece_sentence': mapping, 'piece_weight': piece_weights,
                'positions': positions, 'domain': int(doc['domain'] == 'IT'), 'targets': targets,
                'global_sentence': global_ids, 'teacher': np.asarray(teacher, dtype=np.float32)}


# CELL: Padded sentence sequences plus independently batched token pieces
def collate(rows: list[dict]) -> dict[str, torch.Tensor]:
    sentence_count = max(len(r['targets']) for r in rows)
    pieces = [piece for row in rows for piece in row['pieces']]
    length = max(map(len,pieces)); batch = len(rows)
    ids = torch.zeros((len(pieces),length), dtype=torch.long)
    mask = torch.zeros_like(ids, dtype=torch.bool)
    mapping = []; weights = []; targets = torch.full((batch,sentence_count),-100,dtype=torch.long)
    positions = torch.zeros(batch,sentence_count,2); domains = torch.zeros(batch,dtype=torch.long)
    sentence_mask = torch.zeros(batch,sentence_count,dtype=torch.bool)
    teacher = torch.zeros(batch,sentence_count,13); global_ids = torch.full_like(targets,-1)
    cursor = 0
    for b,row in enumerate(rows):
        count = len(row['targets']); sentence_mask[b,:count] = True
        targets[b,:count] = torch.tensor(row['targets']); positions[b,:count] = torch.tensor(row['positions'])
        domains[b] = row['domain']; teacher[b,:count] = torch.from_numpy(np.asarray(row['teacher']))
        global_ids[b,:count] = torch.tensor(row['global_sentence'])
        for piece,slot,weight in zip(row['pieces'],row['piece_sentence'],row['piece_weight']):
            ids[cursor,:len(piece)] = torch.tensor(piece); mask[cursor,:len(piece)] = True
            mapping.append(b*sentence_count+slot); weights.append(weight); cursor += 1
    return {'input_ids': ids, 'attention_mask': mask, 'piece_sentence': torch.tensor(mapping),
            'piece_weight': torch.tensor(weights), 'sentence_mask': sentence_mask, 'positions': positions,
            'domains': domains, 'targets': targets, 'teacher': teacher, 'global_sentence': global_ids}


class BatchPlan:
    """Shuffle all windows once and use masked dummies instead of dropping/repeating tails."""
    def __init__(self, count, cfg, rank, epoch):
        self.cfg, self.rank = cfg, rank
        self.global_size = cfg.world_size * cfg.micro_batch_size * cfg.accumulation_steps
''')


[Markdown Cell]

Importable source part 4

In [18]:
# [Code Cell]
append_module('data.py', r'''        self.steps = math.ceil(count / self.global_size)
        permutation = np.random.default_rng(cfg.seed + epoch).permutation(count)
        self.indices = np.full(self.steps*self.global_size,-1,dtype=np.int64)
        self.indices[:count] = permutation

    def global_indices(self, step):
        return self.indices[step*self.global_size:(step+1)*self.global_size]

    def micro_indices(self, step, micro):
        grid = self.global_indices(step).reshape(self.cfg.accumulation_steps,self.cfg.world_size,self.cfg.micro_batch_size)
        return grid[micro,self.rank].tolist()


def window_normalizers(cache, dataset, weights):
    result = []
    for window in cache['windows']['train']:
        doc = cache['documents']['train'][window['doc']]
        y = np.asarray(dataset.targets[doc['start']+window['owned_start']:doc['start']+window['owned_end']])
        result.append(float(np.asarray(weights)[y].sum()))
    return np.asarray([*result, 0.0])  # Index -1 is the dummy's zero denominator contribution.
''')


[Markdown Cell]

### io_utils.py

Typed configuration, strict serialization and explicit task identity.

[Markdown Cell]

Importable source part 1

In [19]:
# [Code Cell]
append_module('io_utils.py', r'''# CELL: Strict JSON, checksums and atomic writes
from contextlib import contextmanager
import hashlib
import json
import os
from pathlib import Path
import shutil
import uuid

def strict_pairs(pairs: list[tuple[str, object]]) -> dict:
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f'Duplicate JSON key: {key}')
        result[key] = value
    return result

def reject_constant(value: str):
    raise ValueError(f'Nonfinite JSON value: {value}')

def read_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding='utf-8'), object_pairs_hook=strict_pairs,
                      parse_constant=reject_constant)

def records(path: str | Path):
    with Path(path).open(encoding='utf-8') as stream:
        for line_number, line in enumerate(stream, 1):
            if line.strip():
                yield json.loads(line, object_pairs_hook=strict_pairs, parse_constant=reject_constant)

def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

def identity(value: dict) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                     allow_nan=False).encode()).hexdigest()

def atomic_json(path: str | Path, value: dict) -> None:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('w', encoding='utf-8', newline='\n') as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write('\n'); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)
''')


[Markdown Cell]

Importable source part 2

In [20]:
# [Code Cell]
append_module('io_utils.py', r'''
def ensure_space(directory: Path, needed: int, reserve: int) -> None:
    if shutil.disk_usage(directory).free < needed + reserve:
        raise OSError('Insufficient working disk. Start a fresh Kaggle session and attach the saved output; do not delete committed checkpoints.')

# CELL: Exclusive ownership and verified checkpoint copies
@contextmanager
def run_lock(directory: Path):
    """Linux advisory lock released by the OS after a killed Kaggle process."""
    import fcntl
    directory.mkdir(parents=True, exist_ok=True)
    with (directory / '.training.lock').open('a+') as stream:
        try:
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError('This experiment is already running. Do not launch a second training cell.') from error
        try:
            yield
        finally:
            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)

def publish_copy(source: Path, destination: Path) -> None:
    """Publish a source without modifying it; hard-link only when supported."""
    if source.resolve() == destination.resolve():
        return
    temporary = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.partial')
    try:
        os.link(source, temporary)
    except OSError:
        shutil.copyfile(source, temporary)
    with temporary.open('r+b') as stream:
        os.fsync(stream.fileno())
    os.replace(temporary, destination)
''')


[Markdown Cell]

### metrics.py

Typed configuration, strict serialization and explicit task identity.

[Markdown Cell]

Importable source part 1

In [21]:
# [Code Cell]
append_module('metrics.py', r'''# CELL: Exact thirteen-label combined/domain metrics and official-evaluator parity
import subprocess
import sys
from pathlib import Path
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, precision_recall_fscore_support
from .io_utils import atomic_json, read_json


def metrics(gold,prediction,labels):
    if set(gold) != set(prediction):
        raise ValueError('Missing/extra normalized document IDs')
    truth = []; predicted = []
    for doc_id in gold:
        if len(gold[doc_id]) != len(prediction[doc_id]):
            raise ValueError(f'Sentence count/order mismatch: {doc_id}')
        if any(x not in labels for x in [*gold[doc_id],*prediction[doc_id]]):
            raise ValueError('Unknown rhetorical role')
        truth.extend(gold[doc_id]); predicted.extend(prediction[doc_id])
    precision,recall,f1,support = precision_recall_fscore_support(truth,predicted,labels=labels,zero_division=0)
    return {'macro_f1': float(f1.mean()), 'macro_precision': float(precision.mean()), 'macro_recall': float(recall.mean()),
        'sentences': len(truth), 'documents': len(gold),
        'per_label': {name: {'precision': float(p),'recall': float(r),'f1': float(f),'support': int(n)}
                      for name,p,r,f,n in zip(labels,precision,recall,f1,support)},
        'confusion_matrix': confusion_matrix(truth,predicted,labels=labels).tolist()}


def validate_prediction(prediction,rows,labels):
    expected = {r['id']: len(r['text']) for r in rows}
    if set(prediction) != set(expected): raise ValueError('Submission ID mismatch across CL/IT')
    for doc_id,count in expected.items():
        values = prediction[doc_id]
        if not isinstance(values,list) or len(values) != count or any(not isinstance(x,str) or x not in labels for x in values):
            raise ValueError('Invalid ordered sentence labels: ' + doc_id)


def official(prepared,prediction_path,score):
    root = Path(prepared)
    result = subprocess.run([sys.executable,str(root/'evaluate_phase3.py'),
        '--truth-cl',str(root/'dev_truth_CL.json'),'--truth-it',str(root/'dev_truth_IT.json'),
        '--prediction',str(prediction_path),'--labels',str(root/'labels.json')],capture_output=True,text=True,check=True)
    line = next(line for line in result.stdout.splitlines() if line.startswith('Macro-F1:'))
    if abs(float(line.split(':')[1])-score) > 1e-6:
        raise ValueError('Internal F1 disagrees with supplied official evaluator')
    return result.stdout
''')


[Markdown Cell]

### model.py

Mean/last token features from all pieces reconstruct one sentence vector. A task projection, domain/position features and optional packed bidirectional sentence GRU produce thirteen role logits. Gold roles are not model inputs. Padding is excluded from pooling and recurrent context.

[Markdown Cell]

Importable source part 1

In [22]:
# [Code Cell]
append_module('model.py', r'''# CELL: Own foundation -> token pieces -> original sentences -> document-context labels
from dataclasses import replace
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.checkpoint import checkpoint
from .adaptation import adapt
from .parents import parent_class
from .config import RRConfig


class SentenceRoleModel(nn.Module):
    """Ragged P token pieces [P,T] -> padded document-sentence logits [B,S,13].

    All long-sentence pieces contribute to their original sentence. The optional
    BiGRU sees text-derived sentence features, never previous or current gold roles.
    """
    def __init__(self, backbone: nn.Module, width: int, cfg: RRConfig) -> None:
        super().__init__(); self.backbone,self.cfg = backbone,cfg
        self.projection = nn.Sequential(nn.Linear(2*width,cfg.head_width),nn.GELU(),nn.LayerNorm(cfg.head_width))
        self.domain = nn.Embedding(2,16)
        self.features = nn.Linear(cfg.head_width+18,cfg.head_width)
        self.context = nn.GRU(cfg.head_width,cfg.head_width//2,batch_first=True,bidirectional=True) if cfg.document_context else None
        self.norm = nn.LayerNorm(cfg.head_width)
        self.dropout = nn.Dropout(cfg.dropout)
        self.classifier = nn.Linear(cfg.head_width,13)

    def pooled(self, ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """Token pieces [P,T] -> mean/last features [P,2*foundation_width]."""
        if self.cfg.method == 'prompt':
            prefix = self.cfg.prompt_tokens
            ids = F.pad(ids,(prefix,0),value=1)
            mask = F.pad(mask,(prefix,0),value=True)
        else: prefix = 0
        hidden = self.backbone.encode(ids,mask) if self.cfg.family == 'b' else self.backbone.hidden_states(ids,mask)
        if prefix:
            hidden,mask = hidden[:,prefix:],mask[:,prefix:]
        hidden = hidden.float()  # Stable reduction even if a backbone returns FP16 states.
        lengths = mask.sum(1).clamp_min(1)
        mean = (hidden*mask[...,None]).sum(1) / lengths[:,None]
        last = hidden[torch.arange(ids.shape[0],device=ids.device),lengths-1]
        return torch.cat((mean,last),dim=-1)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                piece_sentence: torch.Tensor, piece_weight: torch.Tensor,
                sentence_mask: torch.Tensor, positions: torch.Tensor,
''')


[Markdown Cell]

Importable source part 2

In [23]:
# [Code Cell]
append_module('model.py', r'''                domains: torch.Tensor) -> torch.Tensor:
        """Pieces [P,T], map [P], mask [B,S], positions [B,S,2], domains [B] -> [B,S,13]."""
        pooled = []
        for first in range(0,input_ids.shape[0],self.cfg.piece_batch_size):
            args = (input_ids[first:first+self.cfg.piece_batch_size],attention_mask[first:first+self.cfg.piece_batch_size])
            value = checkpoint(self.pooled,*args,use_reentrant=False) if self.training and self.cfg.checkpoint_encoder else self.pooled(*args)
            pooled.append(self.projection(value))
        values = torch.cat(pooled)
        batch,sentences = sentence_mask.shape
        # Float32 accumulation protects long-sentence weighting under autocast.
        total = torch.zeros(batch*sentences,values.shape[-1],device=values.device,dtype=torch.float32)
        denominator = torch.zeros(batch*sentences,1,device=values.device,dtype=torch.float32)
        total.index_add_(0,piece_sentence,values.float()*piece_weight[:,None])
        denominator.index_add_(0,piece_sentence,piece_weight[:,None])
        hidden = (total / denominator.clamp_min(1)).reshape(batch,sentences,-1)
        domain = self.domain(domains)[:,None].expand(-1,sentences,-1)
        hidden = F.gelu(self.features(torch.cat((hidden,domain,positions),dim=-1)))
        if self.context is not None:
            lengths = sentence_mask.sum(1).to('cpu',dtype=torch.long)
            packed = pack_padded_sequence(hidden,lengths,batch_first=True,enforce_sorted=False)
            packed,_ = self.context(packed)
            context,_ = pad_packed_sequence(packed,batch_first=True,total_length=sentences)
            hidden = hidden + context
        return self.classifier(self.dropout(self.norm(hidden)))


def forward(model: nn.Module, batch: dict, device: torch.device) -> torch.Tensor:
    keys = ('input_ids','attention_mask','piece_sentence','piece_weight','sentence_mask','positions','domains')
    return model(**{k: batch[k].to(device,non_blocking=True) for k in keys})


def build_model(job: dict, cfg: RRConfig, device: torch.device, load_parent: bool = True) -> SentenceRoleModel:
    Config,Model = parent_class(job['parent_source_dir'],cfg.family,job['parent_contract'])
    base_cfg = replace(Config(**job['parent']['config']),activation_checkpointing=cfg.activation_checkpointing)
    with torch.device('meta'):
        backbone = Model(base_cfg)
    if load_parent:
        path = Path(job['parent']['checkpoint'])
        if path.stat().st_size != job['parent']['checkpoint_bytes'] or path.stat().st_mtime_ns != job['parent']['checkpoint_mtime_ns']:
            raise ValueError('Verified foundation checkpoint changed before loading')
        payload = torch.load(path,map_location='cpu',weights_only=True,mmap=True)
        backbone.load_state_dict(payload['model'],strict=True,assign=True); del payload
    else:
        backbone._apply(lambda tensor: torch.empty_like(tensor,device=device) if tensor.is_meta else tensor.to(device))
    foundation = sum(p.numel() for p in backbone.parameters())
    assert 50_000_000 < foundation < 300_000_000
    capacity = base_cfg.max_source_tokens if cfg.family == 'b' else base_cfg.max_seq_tokens
    if cfg.sentence_tokens + (cfg.prompt_tokens if cfg.method == 'prompt' else 0) > capacity:
''')


[Markdown Cell]

Importable source part 3

In [24]:
# [Code Cell]
append_module('model.py', r'''        raise ValueError('Piece plus soft prompt exceeds the actual foundation context capacity')
    changed = adapt(backbone,cfg)
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(cfg.seed+1000)
        model = SentenceRoleModel(backbone,base_cfg.d_model,cfg).to(device)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert 50_000_000 < total < 300_000_000 and trainable > 0
    model.parameter_report = {'foundation_unique_parameters': foundation, 'total_unique_parameters': total,
        'trainable_parameters': trainable, 'adapted_modules': changed,
        'inactive_B_decoder': cfg.family == 'b'}
    return model
''')


[Markdown Cell]

### parents.py

Load only the exact audited own architecture sources and completed Phase 1 weights. The token-budget name is not parameter count. Report foundation/total/trainable counts separately; PEFT freezes most weights.

[Markdown Cell]

Importable source part 1

In [25]:
# [Code Cell]
append_module('parents.py', r'''# CELL: Match a parent by source and data rather than a fixed epoch count
import hashlib
import importlib
import importlib.util
import json
import math
from pathlib import Path
import sys
import torch
from tokenizers import Tokenizer
from .io_utils import atomic_json, read_json, sha256_file

def source_hashes(directory: Path) -> dict[str,str]:
    return {path.name: sha256_file(path) for path in sorted(directory.glob('*.py'))}

def learning_choices(values: dict, contract: dict) -> dict:
    return {key:value for key,value in values.items() if key not in contract['fingerprint_exclusions']}

def reproduce_parent(values: dict, contract: dict, version: str) -> str:
    payload={'version':contract['runtime_version'],'config':learning_choices(values,contract),
             'data':contract['data_fingerprint'],'sources':contract['source_sha256'],'torch':version}
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()

def intact_pairs(root: Path) -> tuple[list, list]:
    """Check receipts/checkpoint bytes incrementally before mmap loading."""
    pairs=[]; warnings=[]
    for name in ('slot_0.pt','slot_1.pt','best.pt'):
        path=root/'checkpoints'/name; receipt_path=path.with_name(name+'.json')
        if not path.exists() and not receipt_path.exists():
            continue
        try:
            receipt=read_json(receipt_path)
            if any(type(receipt.get(key)) is not int or receipt[key]<0 for key in ('bytes','epoch','step','generation')):
                raise ValueError('Invalid receipt counters')
            if receipt['bytes']!=path.stat().st_size or receipt['sha256']!=sha256_file(path):
                raise ValueError('Size/SHA-256 mismatch')
            pairs.append((path,receipt))
        except (ValueError,OSError,KeyError,TypeError) as error:
            warnings.append(f'{path}: {error}')
    if not pairs:
        raise FileNotFoundError('No intact full checkpoint/receipt pair. ' + '; '.join(warnings))
    return pairs,warnings

def parent_candidates(cfg, contract: dict) -> list[dict]:
    """Return useful rejection reasons for every discovered run root."""
    roots=[Path(cfg.parent_run_dir)] if cfg.parent_run_dir else sorted({p.parent for p in Path(cfg.input_root).rglob('train_config.json')})
    rows=[]
    for root in roots:
''')


[Markdown Cell]

Importable source part 2

In [26]:
# [Code Cell]
append_module('parents.py', r'''        row={'run_dir':str(root),'reasons':[],'accepted':False}
        try:
            required=('train_config.json','data_report.json','training_summary.json',f'code/model_{cfg.family}/model.py')
            missing=[name for name in required if not (root/name).is_file()]
            if missing:
                row['reasons'].append('Missing: '+', '.join(missing))
            else:
                saved=read_json(root/'train_config.json'); summary=read_json(root/'training_summary.json')
                row.update(planned_epochs=saved.get('epochs'),completed_epochs=summary.get('epoch'),step=summary.get('step'),status=summary.get('status'))
                if type(saved.get('epochs')) is not int or saved['epochs']<1:
                    row['reasons'].append('Invalid configured epoch count')
                actual=source_hashes(root/f'code/model_{cfg.family}')
                changed=sorted(name for name in set(actual)|set(contract['source_sha256']) if actual.get(name)!=contract['source_sha256'].get(name))
                if changed:
                    row['reasons'].append('Original source differs: '+', '.join(changed))
                if read_json(root/'data_report.json').get('data_fingerprint')!=contract['data_fingerprint']:
                    row['reasons'].append('Wrong preprocessing identity for this model')
                names=[name for name in ('slot_0.pt','slot_1.pt','best.pt') if (root/'checkpoints'/name).is_file()
                       and (root/'checkpoints'/(name+'.json')).is_file()]
                row['checkpoint_pairs']=names
                if not names:
                    row['reasons'].append('Attach saved output with real .pt files and .pt.json receipts; .ipynb/prediction ZIP is insufficient')
                row['accepted']=not row['reasons']
        except (ValueError,OSError,KeyError,TypeError,AttributeError) as error:
            row['reasons'].append(f'{type(error).__name__}: {error}')
        rows.append(row)
    return rows

# CELL: Check real cursor/config/Torch identity before selecting weights
def validate_payload(payload: dict, receipt: dict, values: dict, contract: dict) -> None:
    required={'model','config','cursor','optimizer','scaler','scheduler','rng_by_rank','world_size'}
    if required-payload.keys():
        raise ValueError('Parent checkpoint lacks full recovery state')
    cursor=payload['cursor']
    if (any(type(cursor.get(key)) is not int or cursor[key]<0 for key in ('epoch','next_batch','global_step','optimizer_steps'))
            or payload['fingerprint']!=receipt['fingerprint'] or payload['step']!=receipt['step']
            or payload['epoch']!=receipt['epoch'] or payload['epoch']!=cursor['epoch']
            or payload['step']!=cursor['global_step'] or cursor['epoch']>values['epochs']
            or cursor['optimizer_steps']>cursor['global_step']
            or payload['scheduler']['steps']!=cursor['optimizer_steps']
            or payload['world_size']!=values['world_size'] or len(payload['rng_by_rank'])!=values['world_size']
            or learning_choices(payload['config'],contract)!=learning_choices(values,contract)):
        raise ValueError('Parent cursor/config/scheduler/RNG/receipt disagree')

def verify_parent(cfg, contract: dict) -> tuple[Path,dict]:
    diagnostics=parent_candidates(cfg,contract)
    atomic_json(Path(cfg.run_dir)/'parent_input_report.json',{'candidates':diagnostics})
    print(json.dumps({'parent_candidates':diagnostics},indent=2),flush=True)
''')


[Markdown Cell]

Importable source part 3

In [27]:
# [Code Cell]
append_module('parents.py', r'''    accepted=[Path(row['run_dir']) for row in diagnostics if row['accepted']]
    if len(accepted)!=1:
        raise ValueError(f'Found {len(accepted)} matching Model {cfg.family.upper()} outputs. Attach ONE full saved '
                         'output or set parent_run_dir to its exact printed run_dir; see parent_input_report.json.')
    root=accepted[0]; values=read_json(root/'train_config.json')
    pairs,warnings=intact_pairs(root)
    latest=max(pairs,key=lambda item:(item[1]['step'],item[1]['generation']))
    payload=torch.load(latest[0],map_location='cpu',weights_only=True,mmap=True)
    validate_payload(payload,latest[1],values,contract)
    versions=list(contract['recorded_torch_candidates'])+[str(torch.__version__)]
    for name in ('continuation_runtime_identity.json','training_runtime_identity.json'):
        path=root/name
        if path.is_file():
            record=read_json(path)
            if isinstance(record.get('torch'),str):
                versions.insert(0,record['torch'])
    matched=next((v for v in dict.fromkeys(versions) if reproduce_parent(values,contract,v)==payload['fingerprint']),None)
    if matched is None:
        raise ValueError('Cannot reproduce original config/source/data/Torch fingerprint. Attach the matching original runtime and tokenizer; do not disable this check.')
    cursor=payload['cursor']
    if cursor['epoch']!=values['epochs'] or cursor['next_batch']!=0:
        raise ValueError(f"Parent has {cursor['epoch']}/{values['epochs']} completed epochs, next_batch={cursor['next_batch']}. Finish/save its continuation before starting a new NER experiment.")
    report=read_json(root/'data_report.json')
    examples=report.get('train_examples')
    coverage=(isinstance(report.get('eligible_source_tokens'),dict) if cfg.family=='b' else report.get('full_window_coverage') is True)
    if type(examples) is not int or examples<1 or not coverage:
        raise ValueError('Parent data report lacks a positive exhaustive training-window count')
    global_batch=values['world_size']*values['micro_batch_size']*values['accumulation_steps']
    expected_steps=math.ceil(examples/global_batch)*cursor['epoch']
    if cursor['global_step']!=expected_steps:
        raise ValueError(f"Parent counters disagree with its exhaustive window plan: expected {expected_steps} boundaries, got {cursor['global_step']}")
    summary=read_json(root/'training_summary.json')
    if summary.get('status')!='complete' or summary.get('epoch')!=cursor['epoch'] or summary.get('step')!=cursor['global_step'] or summary.get('fingerprint')!=payload['fingerprint']:
        raise ValueError('Parent training summary disagrees with its latest intact checkpoint')
    chosen=next((item for item in pairs if item[0].name=='best.pt'),latest) if cfg.parent_checkpoint=='best' else latest
    selected=torch.load(chosen[0],map_location='cpu',weights_only=True,mmap=True)
    validate_payload(selected,chosen[1],values,contract)
    if selected['fingerprint']!=payload['fingerprint']:
        raise ValueError('Selected parent best/latest belongs to another run')
    for name,tensor in selected['model'].items():
        if tensor.is_floating_point():
            if tensor.dtype!=torch.float32 or not all(bool(torch.isfinite(c).all()) for c in tensor.reshape(-1).split(1_000_000)):
                raise ValueError(f'Nonfinite/non-FP32 master weights: {name}')
    result={'family':cfg.family,'config':values,'completed_epochs':cursor['epoch'],'planned_epochs':values['epochs'],
            'checkpoint':str(chosen[0]),'checkpoint_sha256':chosen[1]['sha256'],'checkpoint_bytes':chosen[1]['bytes'],
            'checkpoint_mtime_ns':chosen[0].stat().st_mtime_ns,'checkpoint_epoch':chosen[1]['epoch'],'checkpoint_step':chosen[1]['step'],
            'selection':cfg.parent_checkpoint,'original_fingerprint':payload['fingerprint'],'original_torch':matched,
            'data_fingerprint':contract['data_fingerprint'],'source_sha256':contract['source_sha256'],'checkpoint_warnings':warnings}
''')


[Markdown Cell]

Importable source part 4

In [28]:
# [Code Cell]
append_module('parents.py', r'''    print(f"Verified Model {cfg.family.upper()}: {cursor['epoch']} completed epochs; selected {chosen[0].name} at epoch {chosen[1]['epoch']}",flush=True)
    return root,result

# CELL: Find exactly matching small preprocessing artifacts without reading LM binaries
def tokenizer_inputs(cfg, contract: dict) -> tuple[Path,Path,dict]:
    metadata=[Path(cfg.metadata_path)] if cfg.metadata_path else sorted(Path(cfg.input_root).rglob('metadata.json'))
    matches=[]
    required=('train.bin','val.bin','tokenizer.json','train_documents.jsonl','validation_documents.jsonl','validation_pairs.jsonl')
    for path in metadata:
        if not path.is_file():
            continue
        try:
            value=read_json(path)
            hashes={name:value['artifacts'][name]['sha256'] for name in required}
            hashes['metadata.json']=sha256_file(path)
            fingerprint=hashlib.sha256(json.dumps(hashes,sort_keys=True).encode()).hexdigest()
            if fingerprint==contract['data_fingerprint']:
                matches.append((path,value))
        except (KeyError,TypeError,ValueError,OSError):
            continue
    if not matches:
        raise FileNotFoundError(f'Missing matching Model {cfg.family.upper()} metadata.json. Use its original preprocessing output, not another token-budget export; large .bin files are unnecessary.')
    meta,value=matches[0]; expected=value['artifacts']['tokenizer.json']['sha256']
    candidates=[Path(cfg.tokenizer_path)] if cfg.tokenizer_path else [meta.parent/'tokenizer.json',*Path(cfg.input_root).rglob('tokenizer.json')]
    token=next((p for p in candidates if p.is_file() and sha256_file(p)==expected),None)
    if token is None:
        raise FileNotFoundError('No tokenizer.json matches the verified metadata; attach the original tokenizer')
    tokenizer=Tokenizer.from_file(str(token))
    specials={'<pad>':0,'<s>':1,'</s>':2,'<unk>':3,'<sep>':4,'<mask>':5}
    if tokenizer.get_vocab_size()!=32768 or any(tokenizer.token_to_id(k)!=v for k,v in specials.items()):
        raise ValueError('Unexpected parent tokenizer vocabulary/special IDs')
    return token,meta,{'tokenizer_sha256':expected,'metadata_sha256':sha256_file(meta)}

def parent_class(directory: str, family: str, contract: dict):
    """Import only the checksum-verified original config and model implementation."""
    root=Path(directory)
    if source_hashes(root)!=contract['source_sha256']:
        raise ValueError('Parent source changed after verification')
    package='verified_parent_'+family+'_'+contract['source_sha256']['model.py'][:12]
    if package not in sys.modules:
        spec=importlib.util.spec_from_file_location(package,root/'__init__.py',submodule_search_locations=[str(root)])
        module=importlib.util.module_from_spec(spec); sys.modules[package]=module; spec.loader.exec_module(module)
    config=importlib.import_module(package+'.config'); model=importlib.import_module(package+'.model')
    return config.TrainConfig,getattr(model,contract['class_name'])
''')


[Markdown Cell]

### predict.py

Reload best real task tensors strictly and reproduce development F1 before exporting. Preserve each document ID and sentence order. The final ZIP has exactly one root prediction.json; native per-domain/class results and logits remain available for audit.

[Markdown Cell]

Importable source part 1

In [29]:
# [Code Cell]
append_module('predict.py', r'''# CELL: Deterministic exact sentence coverage; no inference-time label heuristics
import argparse
from dataclasses import asdict, replace
from pathlib import Path
import json
import os
import zipfile
import numpy as np
import torch
from tqdm.auto import tqdm
from .config import RRConfig
from .data import SentenceWindows, collate, cache_data
from .io_utils import atomic_json, read_json, sha256_file
from .metrics import metrics, validate_prediction, official
from .model import build_model, forward
from .parents import source_hashes
from .state import Checkpoints


@torch.inference_mode()
def logits_partition(model,job,cfg,split,device,rank=0,world_size=1):
    """Return document logits in original order; each owned sentence is assigned once."""
    dataset = SentenceWindows(job['cache'],split,cfg)
    docs = job['cache']['documents'][split]
    output = {d['id']: np.full((d['sentences'],13),np.nan,dtype=np.float32)
              for i,d in enumerate(docs) if i % world_size == rank}
    seen = {key: np.zeros(len(value),dtype=np.int64) for key,value in output.items()}
    indices = [i for i,w in enumerate(job['cache']['windows'][split]) if w['doc'] % world_size == rank]
    model.eval()
    for first in tqdm(range(0,len(indices),cfg.micro_batch_size),disable=rank!=0,desc='Predict ' + split,leave=False):
        selected = indices[first:first+cfg.micro_batch_size]
        batch = collate([dataset[i] for i in selected])
        with torch.autocast(device.type,dtype=torch.float16,enabled=device.type=='cuda'):
            logits = forward(model,batch,device).float().cpu().numpy()
        for b,index in enumerate(selected):
            w = job['cache']['windows'][split][index]; doc = docs[w['doc']]
            lo,hi = w['owned_start'],w['owned_end']
            output[doc['id']][lo:hi] = logits[b,lo-w['lo']:hi-w['lo']]
            seen[doc['id']][lo:hi] += 1
    for key,value in output.items():
        if not np.isfinite(value).all() or not (seen[key] == 1).all():
            raise ValueError('Missing/repeated/nonfinite sentence inference: ' + key)
    return output


def labels_from_logits(logits,labels):
    return {key: [labels[int(x)] for x in value.argmax(-1)] if len(value) else [] for key,value in logits.items()}

''')


[Markdown Cell]

Importable source part 2

In [30]:
# [Code Cell]
append_module('predict.py', r'''
def score_dev(logits,prepared,labels):
    prediction = labels_from_logits(logits,labels)
    gold = {}; domains = {}
    for domain in ('CL','IT'):
        current = read_json(Path(prepared)/('dev_truth_'+domain+'.json'))
        gold.update(current)
        domains[domain] = metrics(current,{key: prediction[key] for key in current},labels)
    return {**metrics(gold,prediction,labels),'domains': domains},prediction


# CELL: Strict saved-task relocation for exports, teachers and optional finalization
def load_task(root,prepared,device,workspace=None):
    root = Path(root); job = read_json(root/'job.json')
    if job['version'] != 'cs781-rr-v1' or job['runtime_source_sha256'] != source_hashes(Path(__file__).parent):
        raise ValueError('Saved task requires its exact Phase 3 runtime')
    if source_hashes(root/'code/rr') != job['runtime_source_sha256']:
        raise ValueError('Saved task source incomplete/changed')
    if str(torch.__version__) != job['environment']['torch']:
        raise ValueError('Use recorded task PyTorch build: '+job['environment']['torch'])
    from .preparation import prepared_input
    cfg = replace(RRConfig(**job['config']),run_dir=str(workspace or root),prepared_dir=str(prepared))
    manifest = read_json(Path(prepared)/'prepared_manifest.json')
    if manifest['identity'] != job['prepared_identity']:
        raise ValueError('Saved task/prepared identity mismatch')
    source = root/'parent_source'/('model_'+cfg.family)
    token = root/'tokenizer/tokenizer.json'; meta = root/'tokenizer/metadata.json'
    if source_hashes(source) != job['parent_contract']['source_sha256'] or sha256_file(token) != job['parent']['tokenizer_sha256'] or sha256_file(meta) != job['parent']['metadata_sha256']:
        raise ValueError('Saved foundation source/tokenizer differs')
    manager = Checkpoints(root,job['fingerprint'],cfg.min_free_bytes)
    selected = manager.best()
    if selected is None: raise ValueError('No validation-best task checkpoint')
    payload = manager.load(selected)
    local = {**job,'parent_source_dir': str(source),'tokenizer_path': str(token),'prepared_dir': str(prepared)}
    local['cache'] = cache_data(prepared,token,cfg)
    model = build_model(local,cfg,device,load_parent=False)
    model.load_state_dict(payload['model'],strict=True); del payload
    return model,local,cfg,selected[1]


def package(prediction,root,name):
    root = Path(root); root.mkdir(parents=True,exist_ok=True)
    atomic_json(root/'prediction.json',prediction)
    path = root/name; temporary = path.with_suffix('.zip.partial')
    with zipfile.ZipFile(temporary,'w',zipfile.ZIP_DEFLATED) as archive:
        archive.write(root/'prediction.json','prediction.json')
    with zipfile.ZipFile(temporary) as archive:
        if archive.namelist() != ['prediction.json'] or archive.testzip() is not None:
''')


[Markdown Cell]

Importable source part 3

In [31]:
# [Code Cell]
append_module('predict.py', r'''            raise ValueError('Invalid submission ZIP')
    os.replace(temporary,path)
    return path


def export(job,cfg):
    device = torch.device('cuda',0); torch.set_num_threads(2)
    root = Path(cfg.run_dir)
    model,job,cfg,receipt = load_task(root,job['prepared_dir'],device)
    output = root/'predictions'; output.mkdir(exist_ok=True)
    reports = {}; artifacts = {}
    partitions = ('train','dev','test') if cfg.selection_mode == 'heldout' else ('test',)
    for split in partitions:
        logits = logits_partition(model,job,cfg,split,device)
        prediction = labels_from_logits(logits,job['labels'])
        if split != 'train' or cfg.selection_mode == 'heldout':
            rows = read_json(Path(job['prepared_dir'])/(split+'.json'))
            validate_prediction(prediction,rows,job['labels'])
        atomic_json(output/(split+'_prediction.json'),prediction)
        order = [d['id'] for d in job['cache']['documents'][split]]
        arrays = [logits[key] for key in order]
        flat = np.concatenate(arrays,axis=0) if arrays else np.zeros((0,13),dtype=np.float32)
        np.save(output/(split+'_logits.npy'),flat)
        atomic_json(output/(split+'_order.json'),[{'id': key,'sentences': len(logits[key])} for key in order])
        for suffix in ('_logits.npy','_order.json'):
            name = split+suffix; artifacts[name] = sha256_file(output/name)
        if split == 'dev':
            value,prediction = score_dev(logits,job['prepared_dir'],job['labels'])
            if abs(value['macro_f1']-receipt['validation_macro_f1']) > 1e-6:
                raise ValueError('Reloaded actual best checkpoint does not reproduce its saved dev F1')
            atomic_json(output/'dev_metrics.json',value)
            (output/'official_dev_evaluation.txt').write_text(official(job['prepared_dir'],output/'dev_prediction.json',value['macro_f1']),encoding='utf-8')
            reports['dev'] = value
        if split == 'train':
            gold = {r['id']:r['labels'] for r in rows}
            train_score = metrics(gold,prediction,job['labels'])
            train_score['scope'] = 'In-sample training fit, not held-out performance'
            atomic_json(output/'train_metrics.json',train_score)
    archive = package(read_json(output/'test_prediction.json'),root,'submission_'+cfg.experiment_id+'.zip')
    result = {'status': 'submission_verified','family': cfg.family,'method': cfg.method,
        'experiment_id': cfg.experiment_id,'selection_mode': cfg.selection_mode,'labels': job['labels'],
        'prepared_identity': job['prepared_identity'],'checkpoint_sha256': receipt['sha256'],
        'checkpoint_epoch': receipt['epoch'],'fingerprint': job['fingerprint'],
        'dev_macro_f1': reports.get('dev',{}).get('macro_f1'), 'private_F1': None,
        'parameter_report': model.parameter_report,'logit_artifacts': artifacts,
        'zip_path': str(archive),'zip_sha256': sha256_file(archive),
        'prediction_sha256': sha256_file(root/'prediction.json'), 'test_labels_used': False}
    atomic_json(root/'submission_receipt.json',result)
''')


[Markdown Cell]

Importable source part 4

In [32]:
# [Code Cell]
append_module('predict.py', r'''    del model; torch.cuda.empty_cache()
    return result


def main():
    parser = argparse.ArgumentParser(); parser.add_argument('--job',required=True)
    args = parser.parse_args(); job = read_json(args.job); cfg = RRConfig(**job['config'])
    cfg.validate(); print(json.dumps(export(job,cfg),indent=2),flush=True)


if __name__ == '__main__': main()
''')


[Markdown Cell]

### preparation.py

Download only the seven official Phase 3 JSON files, pin their repository commit and verify ID/label/sentence structures. All released train/dev/test sentences are preserved. Test gold is never downloaded or admitted.

[Markdown Cell]

Importable source part 1

In [33]:
# [Code Cell]
append_module('preparation.py', r'''# CELL: Strict official structures, vocabulary and dataset-only HF download
import argparse
import hashlib
import json
import shutil
from pathlib import Path
import zipfile
from .io_utils import atomic_json, identity, read_json, sha256_file

LABELS = ('Fact','Issue','ArgumentPetitioner','ArgumentRespondent','Statute','Dissent',
          'PrecedentReliedUpon','PrecedentNotReliedUpon','PrecedentOverruled','RulingByLowerCourt',
          'RatioOfTheDecision','RulingByPresentCourt','None')
FILES = ('CL_train.json','IT_train.json','CL_dev.json','IT_dev.json',
         'CL_test.json','IT_test.json','label_vocab.json')


def normal_id(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError('Official IDs must be non-empty strings')
    return value[:-4] if value.endswith('.txt') else value


def vocabulary(value):
    if isinstance(value, list):
        labels = value
    elif isinstance(value, dict) and value:
        labels = list(value) if all(type(x) is int for x in value.values()) else list(value.values())
    else:
        raise ValueError('Unsupported label_vocab.json; expected list or ID mapping')
    if len(labels) != 13 or set(labels) != set(LABELS):
        raise ValueError('Official vocabulary must contain exactly the thirteen supplied RR labels')
    return labels


def documents(value, split, domain):
    """Accept record lists, documented containers, or ID-to-document objects."""
    if isinstance(value, dict) and 'id' in value and 'text' in value:
        value = [value]
    elif isinstance(value, dict):
        for key in ('documents','data'):
            if isinstance(value.get(key), list):
                value = value[key]
                break
        else:
            rows = []
            for key, item in value.items():
                if not isinstance(item, dict):
                    raise ValueError('Unexpected document object: inspect the official JSON structure')
''')


[Markdown Cell]

Importable source part 2

In [34]:
# [Code Cell]
append_module('preparation.py', r'''                if 'id' in item and normal_id(item['id']) != normal_id(key):
                    raise ValueError('Outer document ID differs from record ID')
                rows.append({'id': key, **item})
            value = rows
    if not isinstance(value, list):
        raise ValueError('Expected a list of official document records')
    rows = []
    for record in value:
        if not isinstance(record, dict):
            raise ValueError('Document record must be an object')
        doc_id = normal_id(record.get('id'))
        sentences = record.get('text')
        if not isinstance(sentences, list) or any(not isinstance(s, str) for s in sentences):
            raise ValueError(f'{doc_id}: text must be an ordered list of sentence strings')
        row = {'id': doc_id, 'domain': domain, 'text': sentences}
        if split != 'test':
            labels = record.get('labels')
            if not isinstance(labels, list) or len(labels) != len(sentences) or any(x not in LABELS for x in labels):
                raise ValueError(f'{doc_id}: missing/invalid labels or sentence-count mismatch')
            row['labels'] = labels
        elif 'labels' in record:
            raise ValueError('Test annotations found; attach only the official unlabelled test input')
        rows.append(row)
    return rows


def download_official(repo_id, revision, destination, token, path_map=None):
    """Pin one commit and fetch just seven small RR JSON files; never a snapshot."""
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=token)
    try:
        commit = api.dataset_info(repo_id, revision=revision).sha
        names = api.list_repo_files(repo_id, repo_type='dataset', revision=commit)
    except Exception as error:
        raise RuntimeError('Cannot access the official dataset: accept its HF access terms, enable Kaggle Internet and the HF_TOKEN/HF Secret.') from error
    selected = {}
    for basename in FILES:
        if path_map and basename in path_map:
            candidates = [path_map[basename]]
        else:
            candidates = [n for n in names if Path(n).name == basename]
            if len(candidates) > 1:
                candidates = [n for n in candidates if 'phase3' in ''.join(c for c in n.lower() if c.isalnum())]
        if len(candidates) != 1 or candidates[0] not in names:
            raise ValueError(f'Cannot uniquely locate {basename}; set HF_PATH_MAP from the repository file listing')
        selected[basename] = candidates[0]
    output = Path(destination); output.mkdir(parents=True, exist_ok=True)
    local = {}
''')


[Markdown Cell]

Importable source part 3

In [35]:
# [Code Cell]
append_module('preparation.py', r'''    for basename, remote in selected.items():
        local[basename] = Path(hf_hub_download(repo_id=repo_id, repo_type='dataset', revision=commit,
            filename=remote, token=token, local_dir=str(output)))
    return local, {'repository': repo_id, 'commit': commit, 'paths': selected}


# CELL: Audit all official documents before publishing prepared artifacts
def prepare(paths, output, provenance, evaluator):
    output = Path(output); output.mkdir(parents=True, exist_ok=True)
    old = output / 'prepared_manifest.json'
    if old.exists():
        existing = read_json(old)
        if existing.get('source_sha256') != {name: sha256_file(path) for name,path in paths.items()} or existing.get('provenance') != provenance:
            raise ValueError('Prepared source/revision changed; use a new output directory before writing')
        for name,receipt in existing['artifacts'].items():
            if sha256_file(output/name) != receipt['sha256']:
                raise ValueError('Existing prepared data changed; restore it before rerunning')
        return existing
    labels = vocabulary(read_json(paths['label_vocab.json']))
    partitions = {}; counts = {}; text_splits = {}; warnings = []
    for split in ('train','dev','test'):
        rows = []
        for domain in ('CL','IT'):
            current = documents(read_json(paths[f'{domain}_{split}.json']), split, domain)
            if not current or sum(len(r['text']) for r in current) < 1:
                raise ValueError(f'{domain}/{split}: no usable official sentences')
            counts[f'{domain}_{split}'] = {'documents': len(current), 'sentences': sum(len(r['text']) for r in current)}
            rows.extend(current)
        if len({r['id'] for r in rows}) != len(rows):
            raise ValueError(f'{split}: ID collision after .txt normalization across CL/IT')
        partitions[split] = rows
        for row in rows:
            if not any(s.strip() for s in row['text']):
                continue
            digest = identity(row['text'])
            previous = text_splits.get(digest)
            if previous and previous != split:
                raise ValueError(f'Full-document text overlaps {previous}/{split}: {row["id"]}; investigate official inputs before training')
            text_splits[digest] = split
    frequency = {label: 0 for label in labels}
    for row in partitions['train']:
        for label in row['labels']:
            frequency[label] += 1
    warnings += [f'Train label absent: {label}' for label, count in frequency.items() if not count]
    for split, rows in partitions.items():
        atomic_json(output / (split + '.json'), rows)
    atomic_json(output / 'labels.json', labels)
    for domain in ('CL','IT'):
''')


[Markdown Cell]

Importable source part 4

In [36]:
# [Code Cell]
append_module('preparation.py', r'''        atomic_json(output / ('dev_truth_' + domain + '.json'), {
            r['id']: r['labels'] for r in partitions['dev'] if r['domain'] == domain})
    shutil.copyfile(evaluator, output / 'evaluate_phase3.py')
    artifacts = {p.name: {'sha256': sha256_file(p), 'bytes': p.stat().st_size} for p in output.iterdir()
                 if p.name in ('train.json','dev.json','test.json','labels.json','dev_truth_CL.json','dev_truth_IT.json','evaluate_phase3.py')}
    manifest = {'version': 1, 'status': 'complete', 'labels': labels, 'counts': counts,
        'train_label_counts': frequency, 'warnings': warnings, 'provenance': provenance,
        'source_sha256': {name: sha256_file(path) for name, path in paths.items()}, 'artifacts': artifacts,
        'private_test_labels_used': False, 'sentences_dropped': 0}
    manifest['identity'] = identity(manifest)
    old = output / 'prepared_manifest.json'
    if old.exists() and read_json(old).get('identity') != manifest['identity']:
        raise ValueError('Prepared input changed; use a new preparation output directory')
    atomic_json(old, manifest)
    archive = output.parent / 'phase3_prepared.zip'
    with zipfile.ZipFile(archive.with_suffix('.zip.partial'), 'w', zipfile.ZIP_DEFLATED) as z:
        for name in [*artifacts, 'prepared_manifest.json']:
            z.write(output / name, name)
    archive.with_suffix('.zip.partial').replace(archive)
    return manifest


def prepared_input(cfg):
    roots = [Path(cfg.prepared_dir)] if cfg.prepared_dir else list(Path(cfg.input_root).rglob('prepared_manifest.json'))
    candidates = []
    for root in roots:
        path = root / 'prepared_manifest.json' if root.is_dir() else root
        if path.is_file() and path.suffix == '.json':
            value = read_json(path)
            if value.get('version') == 1 and value.get('status') == 'complete' and set(value.get('labels', [])) == set(LABELS):
                candidates.append(path.parent)
    if not candidates:
        archives = [Path(cfg.prepared_dir)] if cfg.prepared_dir else list(Path(cfg.input_root).rglob('phase3_prepared.zip'))
        for path in archives:
            if not path.is_file() or path.suffix != '.zip':
                continue
            with zipfile.ZipFile(path) as z:
                manifest = json.loads(z.read('prepared_manifest.json'))
                names = [*manifest['artifacts'], 'prepared_manifest.json']
                if len(z.namelist()) != len(set(z.namelist())) or any(Path(n).name != n or '\\' in n for n in names):
                    raise ValueError('Unsafe/duplicate prepared ZIP member')
                destination = Path(cfg.run_dir) / 'prepared_input'; destination.mkdir(parents=True, exist_ok=True)
                for name in names:
                    with z.open(name) as source, (destination / name).open('wb') as target:
                        shutil.copyfileobj(source, target)
                candidates.append(destination)
    if not candidates:
        raise FileNotFoundError('Attach completed Phase 3 notebook 00 output or phase3_prepared.zip; Phase 2 data is a different task')
''')


[Markdown Cell]

Importable source part 5

In [37]:
# [Code Cell]
append_module('preparation.py', r'''    identities = {read_json(root / 'prepared_manifest.json')['identity'] for root in candidates}
    if len(identities) != 1:
        raise ValueError('Multiple different Phase 3 prepared versions attached; set prepared_dir explicitly')
    root = candidates[0]; manifest = read_json(root / 'prepared_manifest.json')
    saved = manifest['identity']; original = {k: v for k, v in manifest.items() if k != 'identity'}
    if identity(original) != saved or manifest['status'] != 'complete' or set(manifest['labels']) != set(LABELS):
        raise ValueError('Prepared manifest identity/labels invalid')
    for name, receipt in manifest['artifacts'].items():
        if Path(name).name != name or '\\' in name or sha256_file(root / name) != receipt['sha256'] or (root / name).stat().st_size != receipt['bytes']:
            raise ValueError('Changed/missing prepared artifact: ' + name)
    return root
''')


[Markdown Cell]

### prepare.py

Verify own complete foundation/source/tokenizer contracts, or restore an exact task state. A corrupted explicit input cannot silently become a fresh random model. The actual full-size starting model must pass shape, finite-value and adaptation-gradient checks.

[Markdown Cell]

Importable source part 1

In [38]:
# [Code Cell]
append_module('prepare.py', r'''# CELL: Completed own foundations, portable task recovery and task-specific identity
from dataclasses import asdict, replace
from pathlib import Path
import shutil
import numpy as np
import torch
from . import VERSION
from .config import RRConfig
from .data import cache_data, SentenceWindows, collate
from .io_utils import atomic_json, identity, read_json, sha256_file
from .parents import verify_parent, tokenizer_inputs, source_hashes, parent_class
from .preparation import prepared_input
from .state import Checkpoints


def require_hardware():
    if not torch.cuda.is_available() or torch.cuda.device_count() != 2 or not torch.distributed.is_nccl_available():
        raise RuntimeError('Select Kaggle GPU T4 x2; exactly two CUDA devices and NCCL are required')
    for rank in range(2):
        name = torch.cuda.get_device_name(rank)
        if 'T4' not in name: raise RuntimeError('Expected T4 x2, found ' + name)
        print('GPU',rank,':',name,flush=True)


def task_candidates(cfg):
    local = Path(cfg.run_dir)/'job.json'
    if local.is_file(): paths = [local]
    elif cfg.resume_run_dir: paths = [Path(cfg.resume_run_dir)/'job.json']
    else: paths = sorted(Path(cfg.input_root).rglob('job.json'))
    result = []
    for path in paths:
        value = read_json(path)
        if value.get('config',{}).get('experiment_id') != cfg.experiment_id:
            if cfg.resume_run_dir: raise ValueError('Explicit resume belongs to a different experiment ID')
            continue
        if value.get('version') != VERSION or value['runtime_source_sha256'] != source_hashes(Path(__file__).parent):
            raise ValueError('Recovery requires the original complete Phase 3 notebook/runtime')
        saved = RRConfig(**value['config'])
        if saved.choices() != cfg.choices():
            raise ValueError('Learning/head/context settings changed. Use a new ID, or keep settings for exact recovery')
        if value['environment']['torch'] != str(torch.__version__):
            raise ValueError('Use saved recovery PyTorch build ' + value['environment']['torch'])
        if source_hashes(path.parent/'code/rr') != value['runtime_source_sha256']:
            raise ValueError('Saved recovery source files are incomplete/changed')
        result.append((path.parent,value))
    if len({job['fingerprint'] for _,job in result}) > 1:
        raise ValueError('Conflicting outputs share this ID; specify exactly one resume_run_dir')
    return result
''')


[Markdown Cell]

Importable source part 2

In [39]:
# [Code Cell]
append_module('prepare.py', r'''

def teacher_inputs(job,cfg):
    """Verify completed own RR teacher and cache its probabilities aligned by sentence IDs."""
    previous = Path(cfg.run_dir)/'teacher_cache.npy'
    receipt_path = Path(cfg.run_dir)/'teacher_reference.json'
    if previous.is_file() and receipt_path.is_file():
        receipt = read_json(receipt_path)
        if sha256_file(previous) != receipt['cache_sha256']: raise ValueError('Teacher cache changed')
        if cfg.required_teacher_sha256 and receipt['teacher_checkpoint_sha256'] != cfg.required_teacher_sha256:
            raise ValueError('Frozen KD recipe requires its original teacher')
        return receipt
    paths = [Path(cfg.teacher_run_dir)/'job.json'] if cfg.teacher_run_dir else list(Path(cfg.input_root).rglob('job.json'))
    candidates = []
    for path in paths:
        value = read_json(path)
        if value.get('version') != VERSION or value['config'].get('family') not in ('a','c','d'):
            continue
        root = path.parent
        if value['config']['selection_mode'] != 'heldout': continue
        if not (root/'submission_receipt.json').is_file() or not (root/'training_summary.json').is_file(): continue
        receipt = read_json(root/'submission_receipt.json')
        if cfg.required_teacher_sha256 and receipt['checkpoint_sha256'] != cfg.required_teacher_sha256: continue
        if read_json(root/'training_summary.json')['status'] != 'complete' or receipt.get('status') != 'submission_verified': continue
        if receipt['prepared_identity'] != job['prepared_identity'] or receipt['labels'] != job['labels']:
            raise ValueError('Teacher uses different official RR data/label order')
        if receipt['parameter_report']['foundation_unique_parameters'] <= job['parent_parameter_count']:
            continue
        candidates.append((receipt['dev_macro_f1'],root,value,receipt))
    if not candidates:
        raise FileNotFoundError('KD requires a complete trained PHASE 3 A/C/D held-out task output (job, best.pt/receipt, train/dev logits). Run notebooks 01/02 first and attach their saved outputs; a Phase 1 or Phase 2 notebook is not an RR teacher.')
    _,root,teacher,receipt = max(candidates,key=lambda r:(r[0],str(r[1])))
    if source_hashes(root/'code/rr') != job['runtime_source_sha256'] or teacher['runtime_source_sha256'] != job['runtime_source_sha256']:
        raise ValueError('Teacher runtime changed')
    best = root/'checkpoints/best.pt'
    if sha256_file(best) != receipt['checkpoint_sha256'] or read_json(best.with_name(best.name+'.json'))['sha256'] != receipt['checkpoint_sha256']:
        raise ValueError('Teacher real checkpoint and receipt disagree')
    combined = []
    partitions = ('train','dev') if cfg.selection_mode == 'refit' else ('train',)
    for split in partitions:
        for suffix in ('_logits.npy','_order.json'):
            name = split+suffix
            if sha256_file(root/'predictions'/name) != receipt['logit_artifacts'][name]:
                raise ValueError('Teacher exported logits/order changed')
        order = read_json(root/'predictions'/(split+'_order.json'))
        expected = read_json(Path(job['prepared_dir'])/(split+'.json'))
        if order != [{'id': r['id'],'sentences': len(r['text'])} for r in expected]:
            raise ValueError('Teacher sentence ordering differs from official preparation')
''')


[Markdown Cell]

Importable source part 3

In [40]:
# [Code Cell]
append_module('prepare.py', r'''        values = np.load(root/'predictions'/(split+'_logits.npy'),allow_pickle=False)
        if values.shape != (sum(len(r['text']) for r in expected),13) or not np.isfinite(values).all():
            raise ValueError('Invalid teacher logits')
        combined.append(values)
    np.save(previous,np.concatenate(combined))
    reference = {'teacher_checkpoint_sha256': receipt['checkpoint_sha256'], 'teacher_experiment': teacher['config']['experiment_id'],
        'teacher_dev_macro_f1': receipt['dev_macro_f1'],'teacher_foundation_parameters': receipt['parameter_report']['foundation_unique_parameters'],
        'student_foundation_parameters': job['parent_parameter_count'],'cache_sha256': sha256_file(previous),
        'teacher_fingerprint': teacher['fingerprint'],'training_only': cfg.selection_mode == 'heldout',
        'private_test_used': False}
    atomic_json(receipt_path,reference)
    return reference


# CELL: Freeze data/source identities and verify complete task recovery states
def prepare_job(cfg,catalog):
    cfg.validate(); run = Path(cfg.run_dir); run.mkdir(parents=True,exist_ok=True)
    prepared = prepared_input(cfg); manifest = read_json(prepared/'prepared_manifest.json')
    jobs = task_candidates(cfg); prior = jobs[0] if jobs else None
    sources = run/'parent_source'/('model_'+cfg.family); sources.mkdir(parents=True,exist_ok=True)
    small = run/'tokenizer'; small.mkdir(exist_ok=True)
    contract = catalog[cfg.family]
    if prior:
        root,saved = prior; parent = saved['parent']
        for name in contract['source_sha256']:
            origin = root/'parent_source'/('model_'+cfg.family)/name
            if sha256_file(origin) != contract['source_sha256'][name]: raise ValueError('Saved foundation source changed')
            if origin.resolve() != (sources/name).resolve(): shutil.copyfile(origin,sources/name)
        for name,key in (('tokenizer.json','tokenizer_sha256'),('metadata.json','metadata_sha256')):
            origin = root/'tokenizer'/name
            if sha256_file(origin) != parent[key]: raise ValueError('Saved tokenizer/metadata changed')
            if origin.resolve() != (small/name).resolve(): shutil.copyfile(origin,small/name)
        # Recovery includes teacher logits so an interrupted KD run need not reattach its teacher.
        for name in ('teacher_cache.npy','teacher_reference.json'):
            if (root/name).is_file() and (root/name).resolve() != (run/name).resolve(): shutil.copyfile(root/name,run/name)
    else:
        root,parent = verify_parent(cfg,contract)
        token,meta,hashes = tokenizer_inputs(cfg,contract); parent.update(hashes)
        for name in contract['source_sha256']: shutil.copyfile(root/('code/model_'+cfg.family)/name,sources/name)
        shutil.copyfile(token,small/'tokenizer.json'); shutil.copyfile(meta,small/'metadata.json')
    if cfg.expected_parent_sha256 and parent['checkpoint_sha256'] != cfg.expected_parent_sha256:
        raise ValueError('Wrong selected Phase 1 checkpoint. Attach the audited current output, or explicitly use None to accept another fully verified own completed continuation.')
    cache = cache_data(prepared,small/'tokenizer.json',cfg)
    ParentConfig,ParentModel = parent_class(str(sources),cfg.family,contract)
    with torch.device('meta'):
        parameter_probe = ParentModel(ParentConfig(**parent['config']))
    foundation_parameters = sum(p.numel() for p in parameter_probe.parameters())
    assert 50_000_000 < foundation_parameters < 300_000_000
''')


[Markdown Cell]

Importable source part 4

In [41]:
# [Code Cell]
append_module('prepare.py', r'''    del parameter_probe
    environment = {'torch': str(torch.__version__),'tokenizers': __import__('tokenizers').__version__, 'numpy': np.__version__}
    source = source_hashes(Path(__file__).parent)
    job = {'version': VERSION, 'config': asdict(cfg),'prepared_dir': str(prepared),'prepared_identity': manifest['identity'],
        'parent': parent,'parent_contract': contract,'parent_source_dir': str(sources),'tokenizer_path': str(small/'tokenizer.json'),
        'runtime_source_sha256': source,'environment': environment,'cache': cache,'labels': manifest['labels'],
        'parent_parameter_count': foundation_parameters,
        'resume_roots': [str(root) for root,_ in jobs]}
    if cfg.method == 'kd': job['teacher'] = teacher_inputs(job,cfg)
    fingerprint = identity({'version': VERSION,'choices': cfg.choices(),'prepared': manifest['identity'],
        'parent': parent['checkpoint_sha256'],'runtime': source,'environment': environment,'cache': cache['identity'],
        'teacher': job.get('teacher')})
    if prior and prior[1]['fingerprint'] != fingerprint:
        raise ValueError('Recovery fingerprint changed: data/tokenizer/dependency/source/teacher mismatch')
    job['fingerprint'] = fingerprint
    manager = Checkpoints(run,fingerprint,cfg.min_free_bytes,job['resume_roots'])
    latest = manager.latest(); extending = False
    effective_max = cfg.max_epochs
    if latest:
        payload = manager.load(latest)
        original_max = payload['config']['max_epochs']
        effective_max = max(original_max,cfg.max_epochs)
        if cfg.target_total_epochs is not None:
            if cfg.target_total_epochs < payload['cursor']['epoch']:
                raise ValueError('target_total_epochs cannot be below already completed epochs')
            effective_max = cfg.target_total_epochs
            extending = (cfg.target_total_epochs > payload['cursor']['epoch'] and
                         cfg.target_total_epochs != payload['config'].get('target_total_epochs'))
        schedule_epochs = prior[1].get('schedule_epochs',original_max)
        if payload['cursor']['complete'] and cfg.target_total_epochs is None and cfg.max_epochs > original_max:
            raise ValueError('For a completed task use target_total_epochs explicitly; do not change only max_epochs')
        del payload
    else: schedule_epochs = cfg.refit_schedule_epochs if cfg.selection_mode == 'refit' else cfg.max_epochs
    cfg = replace(cfg,max_epochs=effective_max)
    job.update(config=asdict(cfg),extension_requested=extending,schedule_epochs=schedule_epochs,
               resume_checkpoint=str(latest[0]) if latest else None)
    atomic_json(run/'job.json',job); atomic_json(run/'task_config.json',asdict(cfg))
    return job


def sanity(job,cfg):
    from .model import build_model,forward
    device = torch.device('cuda',0); torch.set_num_threads(2)
    model = build_model(job,cfg,device,load_parent=not bool(job['resume_checkpoint']))
    if job['resume_checkpoint']:
        payload = torch.load(job['resume_checkpoint'],map_location='cpu',weights_only=True,mmap=True)
        model.load_state_dict(payload['model'],strict=True); del payload
    model.train(); dataset = SentenceWindows(job['cache'],'train',cfg)
''')


[Markdown Cell]

Importable source part 5

In [42]:
# [Code Cell]
append_module('prepare.py', r'''    batch = collate([dataset[0]])
    with torch.autocast('cuda',dtype=torch.float16):
        logits = forward(model,batch,device)
        loss = torch.nn.functional.cross_entropy(logits.float().transpose(1,2),batch['targets'].to(device),ignore_index=-100)
    if logits.shape != (*batch['targets'].shape,13) or not torch.isfinite(loss): raise ValueError('RR shape/finite check failed')
    loss.backward()
    head = model.classifier.weight.grad
    adaptation = [p.grad for p in model.backbone.parameters() if p.requires_grad]
    if head is None or not torch.isfinite(head).all() or not any(g is not None and torch.isfinite(g).all() for g in adaptation):
        raise ValueError('Missing/nonfinite task-head or backbone-adaptation gradients')
    result = {**model.parameter_report,'logit_shape': list(logits.shape),'finite_shape_and_gradients': True,
              'scope': 'Full architecture/real starting tensors sanity, not a trained F1 score'}
    atomic_json(Path(cfg.run_dir)/'sanity_report.json',result)
    del model,logits,loss; torch.cuda.empty_cache()
    return result
''')


[Markdown Cell]

### select.py

Compare actual completed tasks on identical official development data. Equal-weight pairs need a conservative gain and conditional document-level cross-fit gate; otherwise export the best single. Freeze the recipe before opening test logits. Prior dev checkpoint selection limits independence.

[Markdown Cell]

Importable source part 1

In [43]:
# [Code Cell]
append_module('select.py', r'''# CELL: CPU-only actual-output comparison, conditional document cross-fitting and fallback
import argparse
from itertools import combinations
from pathlib import Path
import json
import numpy as np
from .config import RRConfig
from .io_utils import atomic_json, read_json, sha256_file
from .preparation import prepared_input
from .metrics import metrics,validate_prediction,official
from .predict import package
from .workflow import candidates


def read_scores(root,receipt,rows,split):
    directory = Path(root)/'predictions'
    for suffix in ('_logits.npy','_order.json'):
        name = split+suffix
        if sha256_file(directory/name) != receipt['logit_artifacts'][name]:
            raise ValueError('Actual score artifact differs from task receipt: ' + str(directory/name))
    order = read_json(directory/(split+'_order.json'))
    if order != [{'id':row['id'],'sentences':len(row['text'])} for row in rows]:
        raise ValueError('Different document/sentence ordering in cached task scores')
    values = np.load(directory/(split+'_logits.npy'),allow_pickle=False)
    if values.shape != (sum(len(r['text']) for r in rows),13) or not np.isfinite(values).all():
        raise ValueError('Invalid cached RR logits')
    shifted = values-values.max(-1,keepdims=True)
    probability = np.exp(shifted); return probability/probability.sum(-1,keepdims=True)


def prediction(probability,rows,labels):
    output = {}; start = 0
    for row in rows:
        count = len(row['text'])
        output[row['id']] = [labels[int(x)] for x in probability[start:start+count].argmax(-1)]
        start += count
    if start != len(probability): raise ValueError('Score length mismatch')
    validate_prediction(output,rows,labels)
    return output


def choose_recipe(probabilities,rows,labels,counts,margin=0.005):
    """Fixed equal-weight pairs; leave out whole documents when selecting each fold."""
    gold = {r['id']:r['labels'] for r in rows}
    n = len(probabilities)
    recipes = [(i,) for i in range(n)] + [pair for pair in combinations(range(n),2) if sum(counts[i] for i in pair) <= 300_000_000]
    predictions = [prediction(sum(probabilities[i] for i in recipe)/len(recipe),rows,labels) for recipe in recipes]
    fitted = [metrics(gold,value,labels)['macro_f1'] for value in predictions]
''')


[Markdown Cell]

Importable source part 2

In [44]:
# [Code Cell]
append_module('select.py', r'''    baseline_index = max(range(n),key=lambda i:(fitted[i],-i))
    winner = max(range(len(recipes)),key=lambda i:(fitted[i],-len(recipes[i]),-i))
    oof = {}; fold_choices = []
    if len(rows) >= 3:
        for heldout in rows:
            train_gold = {key:value for key,value in gold.items() if key != heldout['id']}
            scores = [metrics(train_gold,{k:p[k] for k in train_gold},labels)['macro_f1'] for p in predictions]
            chosen = max(range(len(recipes)),key=lambda i:(scores[i],-len(recipes[i]),-i))
            oof[heldout['id']] = predictions[chosen][heldout['id']]
            fold_choices.append({'heldout':heldout['id'],'recipe':recipes[chosen]})
        conditional_oof = metrics(gold,oof,labels)['macro_f1']
    else: conditional_oof = -1.0
    ensemble = len(recipes[winner]) == 2 and fitted[winner] >= fitted[baseline_index]+margin and conditional_oof >= fitted[baseline_index]+margin
    selected = winner if ensemble else baseline_index
    return {'recipe':list(recipes[selected]),'ensemble_selected':ensemble,'selected_fitted_dev_F1':fitted[selected],
        'best_single_dev_F1':fitted[baseline_index],'conditional_selection_OOF_F1':conditional_oof,
        'crossfit_scope':'Conditional on checkpoints already selected on dev, not an independent test estimate',
        'minimum_gain':margin,'fold_choices':fold_choices,'test_labels_used':False},predictions[selected]


def execute(plan):
    cfg = RRConfig(**plan['settings']); root = Path(cfg.run_dir); root.mkdir(parents=True,exist_ok=True)
    prepared = prepared_input(cfg); manifest = read_json(prepared/'prepared_manifest.json')
    rows = read_json(prepared/'dev.json'); test_rows = read_json(prepared/'test.json'); labels = manifest['labels']
    options = candidates(cfg.input_root)
    eligible = []
    for score,directory,job,receipt in options:
        if receipt['prepared_identity'] != manifest['identity'] or receipt['labels'] != labels:
            raise ValueError('Attached RR output uses a different data/vocabulary identity')
        if any(old[3]['checkpoint_sha256'] == receipt['checkpoint_sha256'] for old in eligible): continue
        values = read_scores(directory,receipt,rows,'dev')
        recomputed = metrics({r['id']:r['labels'] for r in rows},prediction(values,rows,labels),labels)['macro_f1']
        if abs(recomputed-score)>1e-6: raise ValueError('Saved logits do not reproduce actual receipt dev F1')
        eligible.append((score,directory,job,receipt,values))
    if not eligible: raise FileNotFoundError('Attach at least one COMPLETE held-out Phase 3 training output with its logits and receipt. Refit outputs are deliberately excluded.')
    comparison = [{'experiment': row[2]['config']['experiment_id'],'method':row[2]['config']['method'],
                   'family':row[2]['config']['family'],'dev_macro_f1':row[0],'task_root':str(row[1])} for row in eligible]
    top = eligible[:3]
    result,dev_prediction = choose_recipe([r[4] for r in top],rows,labels,
                                         [r[3]['parameter_report']['total_unique_parameters'] for r in top])
    frozen = {**result,'selected_experiments':[top[i][2]['config']['experiment_id'] for i in result['recipe']],
              'selected_checkpoint_sha256':[top[i][3]['checkpoint_sha256'] for i in result['recipe']],
              'prepared_identity':manifest['identity']}
    prior = root/'selection_recipe.json'
    if prior.exists() and read_json(prior) != frozen: raise ValueError('Attached candidate pool/selection changed; use a new selection experiment ID')
    atomic_json(prior,frozen)  # Freeze selection BEFORE opening any test logits.
    test_scores = [read_scores(top[i][1],top[i][3],test_rows,'test') for i in result['recipe']]
    test_prediction = prediction(sum(test_scores)/len(test_scores),test_rows,labels)
''')


[Markdown Cell]

Importable source part 3

In [45]:
# [Code Cell]
append_module('select.py', r'''    path = package(test_prediction,root,'submission_'+cfg.experiment_id+'.zip')
    atomic_json(root/'selected_dev_prediction.json',dev_prediction)
    scope = official(prepared,root/'selected_dev_prediction.json',result['selected_fitted_dev_F1'])
    (root/'official_selected_dev.txt').write_text(scope,encoding='utf-8')
    duplicated = [row[2]['config']['experiment_id'] for row in eligible
                  if sha256_file(root/'prediction.json') == row[3]['prediction_sha256']]
    report = {'status':'complete','comparison':comparison,**frozen,'submission_zip':str(path),
        'identical_to_existing_predictions':duplicated,'private_F1':None,
        'missing_required_methods':sorted({'lorma','rslora','lora','ia3','prompt','kd'}-{r[2]['config']['method'] for r in eligible})}
    atomic_json(root/'selection_report.json',report)
    return report


def main():
    parser = argparse.ArgumentParser(); parser.add_argument('--plan',required=True)
    args = parser.parse_args(); print(json.dumps(execute(read_json(args.plan)),indent=2))


if __name__ == '__main__': main()
''')


[Markdown Cell]

### state.py

Two atomic recovery slots plus a best snapshot retain model, Adam, scaler, scheduler, cursor and both ranks RNG. Source/data/Torch identities are checked. Corrupt slots may fall back to another intact state; all-invalid checkpoints stop.

[Markdown Cell]

Importable source part 1

In [46]:
# [Code Cell]
append_module('state.py', r'''# CELL: Per-rank reproducibility and successful-update scheduling
import math
import os
from pathlib import Path
import random
import numpy as np
import torch
from .io_utils import atomic_json, ensure_space, publish_copy, read_json, sha256_file

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False

def capture_rng(device: torch.device) -> dict:
    numpy_state = np.random.get_state()
    return {'python':random.getstate(), 'numpy':[numpy_state[0],numpy_state[1].tolist(),*numpy_state[2:]],
            'torch':torch.get_rng_state(), 'cuda':torch.cuda.get_rng_state(device) if device.type=='cuda' else None}

def restore_rng(state: dict, device: torch.device) -> None:
    random.setstate(state['python'])
    value=state['numpy']; np.random.set_state((value[0],np.asarray(value[1],dtype=np.uint32),*value[2:]))
    torch.set_rng_state(state['torch'])
    if device.type=='cuda':
        torch.cuda.set_rng_state(state['cuda'],device)

class WarmupCosine:
    """Schedule advances after successful optimizer updates, including on resume."""
    def __init__(self, optimizer, total_steps: int, warmup_fraction: float, minimum: float):
        self.optimizer, self.total_steps, self.minimum = optimizer,total_steps,minimum
        self.warmup_steps=max(1,int(total_steps*warmup_fraction))
        self.base_lrs=[group['lr'] for group in optimizer.param_groups]; self.steps=0
        self.apply()

    def apply(self) -> None:
        step=self.steps+1
        if step <= self.warmup_steps:
            scale=step/self.warmup_steps
        else:
            progress=min(1.0,(step-self.warmup_steps)/max(1,self.total_steps-self.warmup_steps))
            scale=self.minimum+(1-self.minimum)*0.5*(1+math.cos(math.pi*progress))
        for group,base in zip(self.optimizer.param_groups,self.base_lrs):
            group['lr']=base*scale

    def advance(self) -> None:
        self.steps+=1; self.apply()

''')


[Markdown Cell]

Importable source part 2

In [47]:
# [Code Cell]
append_module('state.py', r'''    def state_dict(self) -> dict:
        return dict(steps=self.steps,total_steps=self.total_steps,warmup_steps=self.warmup_steps,
                    minimum=self.minimum,base_lrs=self.base_lrs)

    def load_state_dict(self, value: dict) -> None:
        for key in ('total_steps','warmup_steps','minimum','base_lrs'):
            if value[key] != getattr(self,key):
                raise ValueError(f'Scheduler plan changed: {key}')
        self.steps=value['steps']; self.apply()

# CELL: Two atomic full-state recovery slots plus a validation-best snapshot
class Checkpoints:
    """Commit receipts last; select the latest intact state, preserving read-only inputs."""
    def __init__(self, run: Path, fingerprint: str, reserve: int, extra_roots: list[str] | None = None):
        self.directory=run/'checkpoints'; self.directory.mkdir(parents=True,exist_ok=True)
        self.fingerprint,self.reserve,self.generation=fingerprint,reserve,0
        self.roots=[self.directory,*[Path(root)/'checkpoints' for root in (extra_roots or [])]]
        self.warnings=[]

    def candidates(self, best_only: bool = False) -> list[tuple[Path,dict]]:
        result,seen=[],False
        for root in self.roots:
            for name in (('best.pt',) if best_only else ('slot_0.pt','slot_1.pt','best.pt')):
                path=root/name; sidecar=root/(name+'.json')
                if not path.exists() and not sidecar.exists():
                    continue
                seen=True
                try:
                    receipt=read_json(sidecar)
                    if receipt.get('fingerprint') != self.fingerprint or receipt.get('version') != 1:
                        raise ValueError('Experiment/source/Torch fingerprint mismatch')
                    if path.stat().st_size != receipt['bytes'] or sha256_file(path) != receipt['sha256']:
                        raise ValueError('Checkpoint size/SHA mismatch')
                    result.append((path,receipt)); self.generation=max(self.generation,receipt['generation'])
                except (ValueError,KeyError,OSError) as error:
                    self.warnings.append(f'{path}: {error}')
        if seen and not result:
            raise ValueError('All visible NER checkpoints failed verification. Refusing to silently restart. '+ '; '.join(self.warnings))
        return result

    def latest(self) -> tuple[Path,dict] | None:
        choices=self.candidates()
        if not choices:
            return None
        winner=max(choices,key=lambda pair:(pair[1]['step'],pair[1]['generation']))
        tied=[receipt['sha256'] for _,receipt in choices if (receipt['step'],receipt['generation'])==
              (winner[1]['step'],winner[1]['generation'])]
        if len(set(tied))>1:
''')


[Markdown Cell]

Importable source part 3

In [48]:
# [Code Cell]
append_module('state.py', r'''            raise ValueError('Conflicting recovery states at the same step. Attach one exact saved version.')
        return winner

    def best(self) -> tuple[Path,dict] | None:
        values=self.candidates(best_only=True)
        return max(values,key=lambda pair:(pair[1]['validation_macro_f1'],pair[1]['step'])) if values else None

    def adopt(self, candidate: tuple[Path,dict], best: bool = False) -> tuple[Path,dict]:
        source,receipt=candidate
        destination=self.directory/('best.pt' if best else f"slot_{receipt['generation']%2}.pt")
        if source.resolve()!=destination.resolve():
            ensure_space(self.directory,source.stat().st_size,self.reserve)
            publish_copy(source,destination)
            if sha256_file(destination)!=receipt['sha256']:
                raise ValueError('Restaged checkpoint checksum mismatch.')
            atomic_json(destination.with_name(destination.name+'.json'),receipt)
        return destination,receipt

    def load(self, candidate: tuple[Path,dict]) -> dict:
        path,receipt=candidate
        self.generation=max(self.generation,receipt['generation'])
        payload=torch.load(path,map_location='cpu',weights_only=True,mmap=True)
        required={'version','fingerprint','config','model','optimizer','scaler','scheduler','epoch','step','cursor','rng_by_rank'}
        if required-payload.keys() or payload['fingerprint']!=self.fingerprint:
            raise ValueError('Incomplete/incompatible NER checkpoint.')
        cursor=payload['cursor']
        if payload['epoch']!=receipt['epoch'] or payload['step']!=receipt['step'] or cursor['global_step']!=payload['step']:
            raise ValueError('Checkpoint cursor and receipt disagree.')
        if payload['scheduler']['steps']!=cursor['optimizer_steps'] or len(payload['rng_by_rank'])!=payload['config']['world_size']:
            raise ValueError('Missing/misaligned scheduler or per-rank RNG state.')
        return payload

    def commit(self, payload: dict, best: bool = False) -> tuple[Path,dict]:
        self.generation+=1
        path=self.directory/f'slot_{self.generation%2}.pt'
        temporary=path.with_name(path.name+'.partial')
        estimate=sum(tensor.numel()*tensor.element_size() for tensor in payload['model'].values())*3+(16<<20)
        ensure_space(self.directory,estimate,self.reserve)
        with temporary.open('wb') as stream:
            torch.save(payload,stream); stream.flush(); os.fsync(stream.fileno())
        os.replace(temporary,path)
        receipt={'version':1,'fingerprint':self.fingerprint,'sha256':sha256_file(path),'bytes':path.stat().st_size,
                 'epoch':payload['epoch'],'step':payload['step'],'generation':self.generation,
                 'validation_macro_f1':payload['cursor']['best_macro_f1']}
        atomic_json(path.with_name(path.name+'.json'),receipt)
        if best:
            publish_copy(path,self.directory/'best.pt')
            atomic_json(self.directory/'best.pt.json',receipt)
''')


[Markdown Cell]

Importable source part 4

In [49]:
# [Code Cell]
append_module('state.py', r'''        return path,receipt
''')


[Markdown Cell]

### train.py

$L=\sum_i w_{y_i}\mathrm{CE}(z_i,y_i)/\sum_i w_{y_i}$. DDP uses a global denominator, masked dummy tail rows and no_sync accumulation. KD adds temperature-scaled teacher/student KL on owned training sentences. Save complete per-rank recovery state at update boundaries.

[Markdown Cell]

Importable source part 1

In [50]:
# [Code Cell]
append_module('train.py', r'''# CELL: Globally normalized loss, deterministic coverage and collective recovery
import argparse
from contextlib import nullcontext
from dataclasses import replace
from datetime import timedelta
import math
import os
from pathlib import Path
import signal
import socket
import time
import numpy as np
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.nn import functional as F
from tqdm.auto import tqdm
from .config import RRConfig
from .data import SentenceWindows, BatchPlan, collate, window_normalizers
from .io_utils import atomic_json, read_json, run_lock
from .model import build_model, forward
from .predict import logits_partition, score_dev
from .metrics import official
from .state import Checkpoints, WarmupCosine, seed_everything, capture_rng, restore_rng


def objective(logits,targets,weights,teacher,cfg):
    flat = targets.reshape(-1); valid = flat != -100
    scores = logits.float().reshape(-1,13)
    supervised = F.cross_entropy(scores,flat,weight=weights,ignore_index=-100,reduction='sum')
    if cfg.method != 'kd': return supervised
    temperature = cfg.distillation_temperature
    soft = F.softmax(teacher.float().reshape(-1,13)/temperature,dim=-1)
    divergence = F.kl_div(F.log_softmax(scores/temperature,dim=-1),soft,reduction='none').sum(-1)*temperature**2
    selected = weights[flat.clamp_min(0)]*valid
    return (1-cfg.distillation_weight)*supervised + cfg.distillation_weight*(divergence*selected).sum()


def initial_cursor():
    return {'epoch': 0,'next_batch': 0,'global_step': 0,'optimizer_steps': 0,'history': [],
            'best_macro_f1': -1.0,'bad_epochs': 0,'complete': False,
            'loss_sum': 0.0,'weight_sum': 0.0,'sentences': 0,'overflows': 0,'consecutive_overflows': 0}


def optimizer_for(model,cfg):
    groups = [{'params': [p for n,p in model.named_parameters() if n.startswith('backbone.') and p.requires_grad], 'lr': cfg.backbone_lr},
              {'params': [p for n,p in model.named_parameters() if not n.startswith('backbone.') and p.requires_grad], 'lr': cfg.head_lr}]
''')


[Markdown Cell]

Importable source part 2

In [51]:
# [Code Cell]
append_module('train.py', r'''    return torch.optim.AdamW(groups,weight_decay=cfg.weight_decay,fused=True)


def save_all(rank,manager,ddp,optimizer,scaler,scheduler,cursor,cfg,job,device,best=False,paused=False):
    states = [None]*cfg.world_size if rank == 0 else None
    dist.gather_object(capture_rng(device),states,dst=0)
    if rank == 0:
        payload = {'version': 1,'fingerprint': job['fingerprint'],'config': job['config'],
            'model': ddp.module.state_dict(),'optimizer': optimizer.state_dict(),'scaler': scaler.state_dict(),
            'scheduler': scheduler.state_dict(),'epoch': cursor['epoch'],'step': cursor['global_step'],
            'cursor': cursor,'rng_by_rank': states}
        _,receipt = manager.commit(payload,best=best)
        atomic_json(Path(cfg.run_dir)/'training_summary.json',{
            'status': 'complete' if cursor['complete'] else ('paused_recoverable' if paused else 'in_progress'),
            'epoch': cursor['epoch'],'next_batch': cursor['next_batch'],'step': cursor['global_step'],
            'optimizer_steps': cursor['optimizer_steps'],'history': cursor['history'],
            'best_dev_macro_f1': cursor['best_macro_f1'] if cfg.selection_mode == 'heldout' else None,
            'checkpoint_sha256': receipt['sha256'],'fingerprint': job['fingerprint']})
    dist.barrier()


def validation(rank,model,job,cfg,device):
    if cfg.selection_mode == 'refit':
        return {'macro_f1': None,'scope': 'Frozen train+dev refit; no held-out evaluation'}
    local = logits_partition(model,job,cfg,'dev',device,rank,cfg.world_size)
    gathered = [None]*cfg.world_size if rank == 0 else None
    dist.gather_object(local,gathered,dst=0)
    value = None
    if rank == 0:
        joined = {key: part for shard in gathered for key,part in shard.items()}
        value,prediction = score_dev(joined,job['prepared_dir'],job['labels'])
        path = Path(cfg.run_dir)/'dev_prediction.json'; atomic_json(path,prediction)
        text = official(job['prepared_dir'],path,value['macro_f1'])
        (Path(cfg.run_dir)/'official_dev_latest.txt').write_text(text,encoding='utf-8')
        atomic_json(Path(cfg.run_dir)/'dev_metrics.json',value)
    wrapper = [value]; dist.broadcast_object_list(wrapper,src=0)
    return wrapper[0]


# CELL: Exact next-batch restart, AMP-safe updates and validation-best checkpoints
def worker(rank,job,port,event):
    cfg = RRConfig(**job['config']); cfg.validate()
    torch.set_num_threads(2); torch.cuda.set_device(rank); device = torch.device('cuda',rank)
    dist.init_process_group('nccl',init_method=f'tcp://127.0.0.1:{port}',rank=rank,world_size=2,timeout=timedelta(minutes=30))
    try:
        seed_everything(cfg.seed)
        manager = Checkpoints(Path(cfg.run_dir),job['fingerprint'],cfg.min_free_bytes,job['resume_roots'])
        latest = (Path(job['resume_checkpoint']),read_json(Path(job['resume_checkpoint']+'.json'))) if job['resume_checkpoint'] else None
''')


[Markdown Cell]

Importable source part 3

In [52]:
# [Code Cell]
append_module('train.py', r'''        payload = manager.load(latest) if latest else None
        model = build_model(job,cfg,device,load_parent=payload is None)
        if payload: model.load_state_dict(payload['model'],strict=True)
        ddp = DDP(model,device_ids=[rank],broadcast_buffers=False,find_unused_parameters=False)
        optimizer = optimizer_for(model,cfg)
        teacher = np.load(Path(cfg.run_dir)/'teacher_cache.npy',mmap_mode='r') if cfg.method == 'kd' else None
        dataset = SentenceWindows(job['cache'],'train',cfg,teacher)
        steps = math.ceil(len(dataset)/(cfg.world_size*cfg.micro_batch_size*cfg.accumulation_steps))
        scheduler = WarmupCosine(optimizer,steps*job['schedule_epochs'],cfg.warmup_fraction,cfg.min_lr_ratio)
        scaler = torch.amp.GradScaler('cuda',init_scale=4096)
        cursor = initial_cursor()
        if payload:
            scheduler.total_steps = payload['scheduler']['total_steps']; scheduler.warmup_steps = payload['scheduler']['warmup_steps']
            scheduler.minimum = payload['scheduler']['minimum']; scheduler.base_lrs = payload['scheduler']['base_lrs']
            optimizer.load_state_dict(payload['optimizer']); scaler.load_state_dict(payload['scaler']); scheduler.load_state_dict(payload['scheduler'])
            cursor = payload['cursor']; restore_rng(payload['rng_by_rank'][rank],device); del payload
            if cursor['global_step'] != cursor['epoch']*steps+cursor['next_batch'] or not 0 <= cursor['next_batch'] <= steps:
                raise ValueError('Recovery cursor differs from exact complete-data batch plan')
        else:
            seed_everything(cfg.seed+rank)
            save_all(rank,manager,ddp,optimizer,scaler,scheduler,cursor,cfg,job,device)
        if job['extension_requested'] and cursor['complete'] and cursor['epoch'] < cfg.max_epochs:
            cursor['complete'] = False; cursor['bad_epochs'] = 0
        if cursor['complete']: return
        weights = torch.tensor(job['cache']['class_weights'],device=device)
        denominators = window_normalizers(job['cache'],dataset,job['cache']['class_weights'])
        expected_sentences = sum(doc['sentences'] for doc in job['cache']['documents']['train'])
        started = time.monotonic()
        while cursor['epoch'] < cfg.max_epochs:
            epoch = cursor['epoch']; plan = BatchPlan(len(dataset),cfg,rank,epoch)
            ddp.train()
            progress = tqdm(total=steps,initial=cursor['next_batch'],disable=rank!=0,mininterval=2,
                            desc=f'{cfg.experiment_id} epoch {epoch+1}/{cfg.max_epochs}')
            for step in range(cursor['next_batch'],steps):
                stop = torch.tensor(int(event.is_set() or time.monotonic()-started >= cfg.max_run_seconds),device=device)
                dist.all_reduce(stop,op=dist.ReduceOp.MAX)
                if stop.item():
                    save_all(rank,manager,ddp,optimizer,scaler,scheduler,cursor,cfg,job,device,paused=True)
                    progress.close(); return
                denominator = float(denominators[plan.global_indices(step)].sum())
                if denominator <= 0: raise ValueError('Zero global target denominator')
                optimizer.zero_grad(set_to_none=True)
                stats = torch.zeros(2,device=device,dtype=torch.float64)
                for micro in range(cfg.accumulation_steps):
                    batch = collate([dataset[index] for index in plan.micro_indices(step,micro)])
                    targets = batch['targets'].to(device,non_blocking=True)
                    context = ddp.no_sync() if micro+1 < cfg.accumulation_steps else nullcontext()
                    with context:
''')


[Markdown Cell]

Importable source part 4

In [53]:
# [Code Cell]
append_module('train.py', r'''                        with torch.autocast('cuda',dtype=torch.float16):
                            logits = forward(ddp,batch,device)
                            loss = objective(logits,targets,weights,batch['teacher'].to(device),cfg)
                        scaler.scale(loss*(cfg.world_size/denominator)).backward()
                    stats[0] += loss.detach().double(); stats[1] += (targets != -100).sum()
                dist.all_reduce(stats)
                if not torch.isfinite(stats).all(): raise FloatingPointError('Nonfinite loss; resume the last committed state')
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(ddp.parameters(),cfg.max_grad_norm)
                old_scale = scaler.get_scale(); scaler.step(optimizer); scaler.update()
                success = scaler.get_scale() >= old_scale
                if success:
                    scheduler.advance(); cursor['optimizer_steps'] += 1; cursor['consecutive_overflows'] = 0
                else:
                    cursor['overflows'] += 1; cursor['consecutive_overflows'] += 1
                if cursor['consecutive_overflows'] > cfg.max_consecutive_overflows:
                    raise FloatingPointError('Repeated AMP overflow; resume committed state and inspect precision/data')
                cursor['global_step'] += 1; cursor['next_batch'] = step+1
                cursor['loss_sum'] += float(stats[0]); cursor['weight_sum'] += denominator; cursor['sentences'] += int(stats[1])
                optimizer.zero_grad(set_to_none=True)
                if rank == 0:
                    progress.update(); progress.set_postfix(loss=cursor['loss_sum']/cursor['weight_sum'],
                        lr=optimizer.param_groups[0]['lr'],step=cursor['global_step'],amp_skips=cursor['overflows'])
                if cursor['global_step'] % 500 == 0:
                    save_all(rank,manager,ddp,optimizer,scaler,scheduler,cursor,cfg,job,device)
            progress.close()
            save_all(rank,manager,ddp,optimizer,scaler,scheduler,cursor,cfg,job,device)
            value = validation(rank,ddp.module,job,cfg,device)
            if cursor['sentences'] != expected_sentences: raise ValueError('Epoch did not supervise every original training sentence exactly once')
            improved = cfg.selection_mode == 'refit' or value['macro_f1'] > cursor['best_macro_f1']+1e-12
            if improved:
                cursor['best_macro_f1'] = value['macro_f1'] if cfg.selection_mode == 'heldout' else -1.0
                cursor['bad_epochs'] = 0
            else: cursor['bad_epochs'] += 1
            cursor['history'].append({'epoch': epoch+1,'weighted_objective': cursor['loss_sum']/cursor['weight_sum'],
                'training_sentences': cursor['sentences'],'overflows': cursor['overflows'],**value})
            cursor.update(epoch=epoch+1,next_batch=0,loss_sum=0.0,weight_sum=0.0,sentences=0,overflows=0)
            cursor['complete'] = cursor['epoch'] >= cfg.max_epochs or (cfg.selection_mode == 'heldout' and
                cursor['epoch'] >= cfg.minimum_epochs and cursor['bad_epochs'] >= cfg.patience)
            save_all(rank,manager,ddp,optimizer,scaler,scheduler,cursor,cfg,job,device,best=improved)
            if rank == 0: print({'epoch': cursor['epoch'],'dev_F1': value['macro_f1'],'complete': cursor['complete']},flush=True)
            if cursor['complete']: return
    finally:
        dist.destroy_process_group()


# CELL: Importable Kaggle two-T4 launch with graceful stop forwarding
def main():
    parser = argparse.ArgumentParser(); parser.add_argument('--job',required=True)
''')


[Markdown Cell]

Importable source part 5

In [54]:
# [Code Cell]
append_module('train.py', r'''    args = parser.parse_args(); job = read_json(args.job); cfg = RRConfig(**job['config'])
    from .prepare import require_hardware
    require_hardware(); cfg.validate()
    os.environ.setdefault('TOKENIZERS_PARALLELISM','false'); os.environ.setdefault('TORCH_NCCL_ASYNC_ERROR_HANDLING','1')
    with run_lock(Path(cfg.run_dir)):
        manager = Checkpoints(Path(cfg.run_dir),job['fingerprint'],cfg.min_free_bytes,job['resume_roots'])
        latest,best = manager.latest(),manager.best()
        if best: manager.adopt(best,best=True)
        if latest:
            latest = manager.adopt(latest); job['resume_checkpoint'] = str(latest[0])
            print('Resume exact saved step',latest[1]['step'],'epoch',latest[1]['epoch'],flush=True)
        else: job['resume_checkpoint'] = None
        context = mp.get_context('spawn'); event = context.Event()
        signal.signal(signal.SIGINT,lambda *_: event.set()); signal.signal(signal.SIGTERM,lambda *_: event.set())
        with socket.socket() as listener:
            listener.bind(('127.0.0.1',0)); port = listener.getsockname()[1]
        processes = mp.spawn(worker,args=(job,port,event),nprocs=2,join=False)
        while not processes.join(timeout=1): pass


if __name__ == '__main__': main()
''')


[Markdown Cell]

### workflow.py

Training, strict replay and frozen train+dev refit are distinct modes. Refit selects a completed held-out task recipe first and does not report a held-out development metric after training on development labels.

[Markdown Cell]

Importable source part 1

In [55]:
# [Code Cell]
append_module('workflow.py', r'''# CELL: Complete task discovery, independent worker processes and evidence packaging
import argparse
from dataclasses import asdict, replace
from pathlib import Path
import json
import os
import signal
import subprocess
import sys
import zipfile
import torch
from .config import RRConfig
from .io_utils import atomic_json, read_json, sha256_file, run_lock
from .parents import source_hashes
from .preparation import prepared_input


def candidates(input_root,explicit=None):
    paths = [Path(explicit)/'job.json'] if explicit else sorted(Path(input_root).rglob('job.json'))
    result = []
    for path in paths:
        value = read_json(path)
        if value.get('version') != 'cs781-rr-v1' or value['config']['selection_mode'] != 'heldout': continue
        receipt_path = path.parent/'submission_receipt.json'; summary = path.parent/'training_summary.json'
        if not receipt_path.is_file() or not summary.is_file(): continue
        receipt = read_json(receipt_path)
        if receipt.get('status') == 'submission_verified' and read_json(summary).get('status') == 'complete':
            if receipt['dev_macro_f1'] is None: continue
            result.append((receipt['dev_macro_f1'],path.parent,value,receipt))
    if explicit and not result: raise ValueError('source_task_dir must be a completed held-out Phase 3 task root containing job.json, actual best.pt and receipt')
    return sorted(result,key=lambda row:(row[0],str(row[1])),reverse=True)


def run_module(module,args,code):
    environment = os.environ.copy(); environment['PYTHONPATH'] = str(code)+os.pathsep+environment.get('PYTHONPATH','')
    process = subprocess.Popen([sys.executable,'-u','-m','rr.'+module,*map(str,args)],cwd=code,env=environment)
    old = {signum: signal.getsignal(signum) for signum in (signal.SIGINT,signal.SIGTERM)}
    def forward_stop(signum,_):
        if process.poll() is None: process.send_signal(signum)
    try:
        for signum in old: signal.signal(signum,forward_stop)
        status = process.wait()
        if status: raise subprocess.CalledProcessError(status,process.args)
    finally:
        for signum,handler in old.items(): signal.signal(signum,handler)


def observations(root):
''')


[Markdown Cell]

Importable source part 2

In [56]:
# [Code Cell]
append_module('workflow.py', r'''    root = Path(root); target = root/'observations.zip'
    names = {'job.json','task_config.json','training_summary.json','sanity_report.json','submission_receipt.json',
             'dev_metrics.json','train_metrics.json','teacher_reference.json','final_report.json','frozen_recipe.json',
             'pipeline_plan.json','prepared_manifest.json'}
    with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
        for path in root.rglob('*.json'):
            if path.name in names: z.write(path,path.relative_to(root))
    return target


# CELL: Train/develop, verified inference, or a frozen train+dev refit
def execute(plan,catalog):
    from .prepare import prepare_job,sanity,require_hardware
    require_hardware()
    cfg = RRConfig(**plan['settings']); cfg.validate(); root = Path(cfg.run_dir)
    prepared = prepared_input(cfg)
    mode = plan.get('mode','train')
    if mode not in ('train','refit','inference'): raise ValueError('Unknown workflow mode')
    if mode in ('refit','inference'):
        choices = candidates(cfg.input_root,plan.get('source_task_dir'))
        if not choices: raise FileNotFoundError('Attach complete trained Phase 3 outputs before selecting/refitting a winner')
        _,source,job,receipt = choices[0]
        if job['runtime_source_sha256'] != source_hashes(Path(__file__).parent) or source_hashes(source/'code/rr') != job['runtime_source_sha256']:
            raise ValueError('Selected task runtime incomplete/changed')
        if receipt['prepared_identity'] != read_json(prepared/'prepared_manifest.json')['identity']:
            raise ValueError('Selected winner uses different Phase 3 data')
        if sha256_file(source/'checkpoints/best.pt') != receipt['checkpoint_sha256']:
            raise ValueError('Winner checkpoint no longer matches its saved receipt')
        selected = RRConfig(**job['config'])
        if mode == 'inference':
            from .predict import load_task,logits_partition,labels_from_logits,score_dev,package
            device = torch.device('cuda',0)
            model,local,selected,_ = load_task(source,prepared,device,root/'inference_cache')
            dev = logits_partition(model,local,selected,'dev',device)
            value,_ = score_dev(dev,prepared,local['labels'])
            if abs(value['macro_f1']-receipt['dev_macro_f1']) > 1e-6: raise ValueError('Replayed selected dev F1 differs')
            logits = logits_partition(model,local,selected,'test',device)
            path = package(labels_from_logits(logits,local['labels']),root,'submission_'+cfg.experiment_id+'.zip')
            result = {'status':'complete','mode':mode,'source':str(source),'dev_macro_f1':value['macro_f1'],
                      'submission_zip':str(path),'private_F1':None}
            atomic_json(root/'final_report.json',result); observations(root); return result
        frozen = {'source_checkpoint_sha256': receipt['checkpoint_sha256'],'source_fingerprint': job['fingerprint'],
            'source_dev_macro_f1': receipt['dev_macro_f1'],'source_best_epoch': receipt['checkpoint_epoch'],
            'choices': selected.choices(),'teacher': job.get('teacher')}
        old = root/'frozen_recipe.json'
        if old.exists() and read_json(old) != frozen: raise ValueError('Refit winner/recipe changed; use a new experiment ID')
        atomic_json(old,frozen)
        cfg = replace(selected,run_dir=str(root),experiment_id=cfg.experiment_id,input_root=cfg.input_root,
''')


[Markdown Cell]

Importable source part 3

In [57]:
# [Code Cell]
append_module('workflow.py', r'''            prepared_dir=str(prepared),parent_run_dir=cfg.parent_run_dir,tokenizer_path=cfg.tokenizer_path,
            metadata_path=cfg.metadata_path,resume_run_dir=cfg.resume_run_dir,teacher_run_dir=cfg.teacher_run_dir,
            required_teacher_sha256=job.get('teacher',{}).get('teacher_checkpoint_sha256'),
            selection_mode='refit',fixed_epochs=receipt['checkpoint_epoch'],max_epochs=receipt['checkpoint_epoch'],
            target_total_epochs=None,refit_schedule_epochs=job['schedule_epochs'],max_run_seconds=cfg.max_run_seconds,
            expected_parent_sha256=job['parent']['checkpoint_sha256'])
    job = prepare_job(cfg,catalog); cfg = RRConfig(**job['config'])
    print(json.dumps({'data_identity':job['prepared_identity'],'parent_completed_epochs':job['parent']['completed_epochs'],
        'family':cfg.family,'method':cfg.method,'mode':cfg.selection_mode},indent=2),flush=True)
    print(json.dumps(sanity(job,cfg),indent=2),flush=True)
    run_module('train',['--job',root/'job.json'],root/'code')
    summary = read_json(root/'training_summary.json')
    if summary['status'] != 'complete':
        result = {'status':'paused_recoverable','submission_zip':None,'private_F1':None,
                  'next_batch':summary['next_batch'],'epoch':summary['epoch'],'step':summary['step']}
    else:
        run_module('predict',['--job',root/'job.json'],root/'code')
        receipt = read_json(root/'submission_receipt.json')
        result = {'status':'complete','mode':cfg.selection_mode,'family':cfg.family,'method':cfg.method,
            'dev_macro_f1':receipt['dev_macro_f1'],'submission_zip':receipt['zip_path'],
            'checkpoint_epoch':receipt['checkpoint_epoch'],'private_F1':None}
    atomic_json(root/'final_report.json',result); observations(root)
    return result


def main():
    parser = argparse.ArgumentParser(); parser.add_argument('--plan',required=True); parser.add_argument('--catalog',required=True)
    args = parser.parse_args(); plan = read_json(args.plan)
    with run_lock(Path(plan['settings']['run_dir'])/'workflow_lock'):
        print(json.dumps(execute(plan,read_json(args.catalog)),indent=2),flush=True)


if __name__ == '__main__': main()
''')


In [58]:
# [Code Cell]
EXPECTED_SOURCE_SHA256={'__init__.py': '330a0c2e5f59d598bf8b4292eb6acdb6a03ebb6868730cb8e5c84c964d4e3e55', 'adaptation.py': 'b6af26e57a97091089d49d6bef24df0f546289dc255c63f3407373c39b316070', 'config.py': '61e3ae2a59cde8509f5ec5cc80e1b308bfc181d5abace9ecfe187f3f566748fe', 'data.py': '7d8be6c3cd4c581aaccc54f8d5e0e5fae9790771b3dbba587a2db6ba4248ee90', 'io_utils.py': '3f9d828342c819690d390f0280c01f9eda160aab59c9cf424d6b3c2a31813e73', 'metrics.py': 'e5281add21ecaee57776c8b955a45b4d5f9dd4c16e51b3b756371d57d09a8f02', 'model.py': '7c99b14f554348ad80f002a9a6bcf8c76b7cc4768585de0b0359e545ef1eb3b2', 'parents.py': '834710b94c3f82eb18b48abf752727e30d245ebbe58f55408263b54fdddcd660', 'predict.py': 'e7cc54f1cfbf1e0ed45e52b7413a18732a8f862ce1b0bd8de8b4281d4df4b520', 'preparation.py': 'd600547d6885b767bd6698ddf3f383fa42da1747772d235c2cd2c4dff2a3b802', 'prepare.py': '6c7b13ab377bba2a515e499da372f43464e7969f8af025635941f4fb30219a1e', 'select.py': '201d7ba35dd5fa9bddcfad9ad88a7fa074005131cb11a9b46a72844674811849', 'state.py': 'f05cc134dcbff7b59050460dccc6d61cbd5b75ad6920175526b459ef3a1eeebf', 'train.py': '511b879d32e52504c25d7310e660874e58f8a50c53aefc848dea5329c8a77b51', 'workflow.py': '3e0893c484dc64cd2226d1eff3ce7427f11dc807b2efda433602185e34f2a9a9'}


[Markdown Cell]

## 4. Materialize exact importable modules

In [59]:
# [Code Cell]
import hashlib
import importlib
code_directory=Path(OUTPUT_ROOT)/'code'
package_directory=code_directory/'rr'
package_directory.mkdir(parents=True,exist_ok=True)
for name,parts in module_parts.items():
    source=''.join(parts)
    assert hashlib.sha256(source.encode()).hexdigest()==EXPECTED_SOURCE_SHA256[name],name
    compile(source,name,'exec')
    destination=package_directory/name
    if (Path(OUTPUT_ROOT)/'job.json').exists() and destination.exists() and destination.read_bytes()!=source.encode():
        raise ValueError('Saved task runtime changed; keep its exact notebook or start a new experiment ID')
    destination.write_bytes(source.encode())
for name in list(sys.modules):
    if name=='rr' or name.startswith('rr.'): del sys.modules[name]
sys.path.insert(0,str(code_directory));importlib.invalidate_caches()


In [60]:
# [Code Cell]
PARENT_CATALOG={'a': {'family': 'a', 'runtime_version': 'cs781-model-a-v1', 'source_sha256': {'config.py': 'c0d1e9e7eb49a11e12ff0d7014cd07c6b90149460da5db3b06e9af71fbb3f390', 'model.py': '05118a8c9eec3e58adb272554e7356e81fb28c95cbe7905f39ebd5ee2d670319', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': '723193f75a39d3b8f16af1e6f1bfbd507b1775a0cd2430583d211e54e69d63c2', 'train.py': '3ea14c27742ab82e686083a669823678529d0f8beb46b2e6b2113f0aef452743', 'infer.py': '1cb8e96f2d49aa1c0b25c8e0ef6420c6d63f17ec8b72813d69b646748fb46013', 'sanity.py': '7716a61318767f9001ac08cdcafbe228f53c3c0d6f8fb9cc3806353ec2d6577f', '__init__.py': '59a82a09276925aa3342bc0991d79d474388cc6119bae880ced9d2403eb20a45'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3571562850214ac1b423548cb0020fdcf1a3ffd8956e0368c456ec1b100292f4', 'audited_original_epochs': 2, 'audited_original_step': 29942, 'audited_original_fingerprint': '24f90bc6e148ffb53dcf6132138b69a95ced2e6a5ca34690601d29def059cddd', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalCausalDecoder'}, 'b': {'family': 'b', 'runtime_version': 'cs781-model-b-v1', 'source_sha256': {'config.py': '71407468d5c4b105b5db2c533fb736eb847b4b67cdf9d4f2ab014436adef1e9b', 'model.py': '09baae8e7ebe0508fb149a64c187975b7a86ae62376b5bc2ec4cef2fb2471a99', 'data.py': 'd01b01d43f5c4c24bb0429f7bbb8bbbd4c52d52c8787bd5ef3f40bdac28df919', 'state.py': '026036ddbdf43e949755da263eb97378855a28281129c6a63f3b3ea69abbe088', 'train.py': 'f928bb52eaafd17dc061d3003326fd13cf2f79b5ccbc226f6af0177429aefab0', 'infer.py': '80ee8fff95675dccb59cf19da091355020495c0f41ea16fdb17d8f24357a59f4', 'sanity.py': '6b1b96bc16a4d7f0945e139fd60f35813b0bf9d2b075c4fd893a87d432fc9212', '__init__.py': '3dfc8da18d4c5d73d0ff1b52dc572672f8095b605dcc86836832d77607749087'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3571562850214ac1b423548cb0020fdcf1a3ffd8956e0368c456ec1b100292f4', 'audited_original_epochs': 3, 'audited_original_step': 24267, 'audited_original_fingerprint': 'e686c0f98fb5fe8fc03553d0151961dd6af0dd52ac041753de0502521a922bb1', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalEncoderDecoder'}, 'c': {'family': 'c', 'runtime_version': 'cs781-model-c-v1', 'source_sha256': {'config.py': 'f9fea0bc87f74ccac761f7d4dc574c4d1315a459523dba21849279c17bc24ace', 'model.py': '6f91835c789d12aadb5355bc06050b8b6c1fc231cd16126fc39f31769706aad4', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': 'b3a3ed7109fdfe43ebc8e75db89f2916d33e95ebe901e99d3e4aa9b6e8422ee4', 'train.py': '9bf6cb023ad1c8fdc4fbb652b4b3b6d3551ae777df139184051f8d23f4e5c0c9', 'infer.py': '062010c87b66c7badcef8f7a62ef3a89fdc7f96149d317214b13a7e679f40b6a', 'sanity.py': 'c56b6ad692b90cb5afb23a03a75a26dde9d9b2c9ef3e14ca2ed37cb8b0f18a68', '__init__.py': '59a82a09276925aa3342bc0991d79d474388cc6119bae880ced9d2403eb20a45'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '092deede43ffd17c00e5d11e1f4aa3a443b7939af4504c488426b70c53d1ed54', 'audited_original_epochs': 1, 'audited_original_step': 7383, 'audited_original_fingerprint': '8febd61da7f1d32f78eee1cd4a865d72e5a5b3a0068739210db323859c6d7caa', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalTransformerGRU'}, 'd': {'family': 'd', 'runtime_version': 'cs781-model-d-v1', 'source_sha256': {'config.py': '42bd096bdf40822a7cb1d84e7f437e3e64d2ef20b99ee81c884f60639378e63a', 'model.py': 'a2300d34524df50cb43a3ff700e72b3a8764f7566d66ba5065c048c5f115eae9', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': 'dee5a29320756f4b84ec349b24091180be65a8f4c86e7f16eb74343737f4e2db', 'train.py': 'efb53d8eb54a08ef1a4abc49d5437eb472ebf5047bf790cf2f041c17fd6cef37', 'infer.py': 'aa3b849b0095bbdb5d81d3bfafe6a8134d4481856ee7cb9db5fba2bd84b80b67', 'sanity.py': '8c02e1c1a1e521a5d1e83fd96b3cfba3fa9f7c76c5824f62d45297a4829ca298', '__init__.py': 'f49e11a2d96bae160f02704fd5818fe613e77534e06a0f16150dbc4bc2d4f5e6'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3a4dc2be3ff16945069b15d7a2e351012d4a0c9319b95b5e4534cc6596e8693f', 'audited_original_epochs': 1, 'audited_original_step': 1828, 'audited_original_fingerprint': 'b9659899c46424ec56ad0607f32bd6e7097752a8066dd30a38764469e05c1352', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalSparseDecoder'}}
catalog_path=code_directory/'parent_catalog.json'
catalog_path.write_text(json.dumps(PARENT_CATALOG,indent=2),encoding='utf-8')


5810

In [61]:
# [Code Cell]
# Select only the exact completed A01 task; do not silently refit a different method.
if SOURCE_TASK_DIR is None:
    matches=[]
    for path in Path('/kaggle/input').rglob('job.json'):
        candidate=json.loads(path.read_text(encoding='utf-8'))
        if candidate.get('config',{}).get('experiment_id')!='P3_01_A_LoRMA': continue
        if not (path.parent/'submission_receipt.json').is_file(): continue
        if not (path.parent/'training_summary.json').is_file(): continue
        if json.loads((path.parent/'training_summary.json').read_text(encoding='utf-8')).get('status')=='complete': matches.append(path.parent)
    if not matches: raise FileNotFoundError('Attach COMPLETE saved A01 output with job.json, actual best.pt/receipt, exported logits and a complete training_summary; finish/resume 01 first.')
    SOURCE_TASK_DIR=str(sorted(matches)[0])
selected_job=json.loads((Path(SOURCE_TASK_DIR)/'job.json').read_text(encoding='utf-8'))
if selected_job.get('config',{}).get('experiment_id')!='P3_01_A_LoRMA': raise ValueError('This compatibility notebook is only for the original A01 task.')
if selected_job.get('runtime_source_sha256')!=EXPECTED_SOURCE_SHA256: raise ValueError('A01 source version differs; use its exact original runtime.')
print('Frozen original-runtime A01 source:',SOURCE_TASK_DIR)


Frozen original-runtime A01 source: /kaggle/input/notebooks/shubhamjoshi1806/01-a-lorma/cs781_phase3/P3_01_A_LoRMA


In [62]:
# [Code Cell]
from rr.config import RRConfig
from rr.io_utils import atomic_json,read_json
SETTINGS=dict(family='a',method='lorma',experiment_id=EXPERIMENT_ID,run_dir=OUTPUT_ROOT,
    prepared_dir=INPUTS['prepared_dir'],parent_run_dir=INPUTS['parent_run_dir'],
    tokenizer_path=INPUTS['tokenizer_path'],metadata_path=INPUTS['metadata_path'],
    resume_run_dir=INPUTS['resume_run_dir'],teacher_run_dir=INPUTS['teacher_run_dir'],
    expected_parent_sha256=EXPECTED_PARENT_SHA256,max_epochs=MAX_EPOCHS,
    target_total_epochs=TARGET_TOTAL_EPOCHS,max_run_seconds=int(TRAINING_BUDGET_MINUTES*60),
    document_context=True,backbone_lr=0.0002,
    head_lr=5e-4,rank=16,alpha=32.0)
RRConfig(**SETTINGS).validate()
PLAN=dict(mode='refit',settings=SETTINGS,source_task_dir=SOURCE_TASK_DIR)
plan_path=Path(OUTPUT_ROOT)/'pipeline_plan.json';atomic_json(plan_path,PLAN)


[Markdown Cell]

## 5. Execute and recover

Training uses two spawned NCCL workers with FP16, rank-0 progress and full epoch/every-500-update checkpoints. Keep the launch cell running after requesting a safe stop. Save complete output for another session. Changing head/LR/context needs a new experiment ID.

In [63]:
# [Code Cell]
import codecs
import select
import signal

def run_streamed(command):
    environment=os.environ.copy()
    environment['PYTHONPATH']=str(code_directory)+os.pathsep+environment.get('PYTHONPATH','')
    process=subprocess.Popen(command,cwd=code_directory,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=0)
    decoder=codecs.getincrementaldecoder('utf-8')(errors='replace')
    assert process.stdout is not None
    try:
        while True:
            try:
                ready,_,_=select.select([process.stdout],[],[],1.0)
                if ready:
                    chunk=os.read(process.stdout.fileno(),8192)
                    if chunk:
                        sys.stdout.write(decoder.decode(chunk));sys.stdout.flush()
                    elif process.poll() is not None: break
                elif process.poll() is not None: break
            except KeyboardInterrupt:
                print('Requesting a saved stop at the next update boundary; keep this cell running.')
                process.send_signal(signal.SIGINT)
        status=process.wait()
        sys.stdout.write(decoder.decode(b'',final=True));sys.stdout.flush()
        if status: raise subprocess.CalledProcessError(status,command)
    finally: process.stdout.close()


In [64]:
# [Code Cell]
run_streamed([sys.executable,'-u','-m','rr.workflow','--plan',str(plan_path),'--catalog',str(catalog_path)])


GPU 0 : Tesla T4
GPU 1 : Tesla T4
{
  "parent_candidates": [
    {
      "run_dir": "/kaggle/input/notebooks/omprakashnetam69/modela-training-final/model_a_extended_run",
      "reasons": [],
      "accepted": true,
      "planned_epochs": 4,
      "completed_epochs": 4,
      "step": 59884,
      "status": "complete",
      "checkpoint_pairs": [
        "slot_0.pt",
        "slot_1.pt",
        "best.pt"
      ]
    }
  ]
}
Verified Model A: 4 completed epochs; selected best.pt at epoch 4
Tokenize test: 100%|██████████| 10/10 [00:00<00:00, 40.58it/s]
{
  "data_identity": "e0d5e0b0be49274a1f96ddbe2c49a1a86709fc691ec2dc82c8874854f6d48758",
  "parent_completed_epochs": 4,
  "family": "a",
  "method": "lorma",
  "mode": "refit"
}
{
  "foundation_unique_parameters": 110119680,
  "total_unique_parameters": 113931821,
  "trainable_parameters": 3812141,
  "adapted_modules": [
    "layers.0.attention.qkv",
    "layers.0.attention.out",
    "layers.0.ffn.gate",
    "layers.0.ffn.up",
    "layer

[Markdown Cell]

## 6. Read the real result and download the exact submission

In [65]:
# [Code Cell]
report=read_json(Path(OUTPUT_ROOT)/'final_report.json')
print(json.dumps(report,indent=2))
from IPython.display import display,FileLink
if report['status']=='complete' and report.get('submission_zip'):
    display(FileLink(report['submission_zip']))
else: print('No completed new submission yet. Save the whole output and resume the SAME notebook/ID/settings.')
for name in ('observations.zip','selection_report.json','training_summary.json','submission_receipt.json'):
    path=Path(OUTPUT_ROOT)/name
    if path.is_file(): display(FileLink(str(path)))
print('Keep the complete checkpoint output. Private F1 is unknown until actual server evaluation.')


{
  "status": "paused_recoverable",
  "submission_zip": null,
  "private_F1": null,
  "next_batch": 218,
  "epoch": 4,
  "step": 1426
}
No completed new submission yet. Save the whole output and resume the SAME notebook/ID/settings.


/kaggle/working/cs781_phase3/P3_08_A01_Original_Runtime_Refit/observations.zip

/kaggle/working/cs781_phase3/P3_08_A01_Original_Runtime_Refit/training_summary.json

Keep the complete checkpoint output. Private F1 is unknown until actual server evaluation.


[Markdown Cell]

## 7. What to keep and do next

The final report identifies the actual output ZIP. Submit only that ZIP, with one root prediction.json. Keep the entire saved output: checkpoints/best.pt and its receipt; slot_0/slot_1 for exact recovery; job.json and task_config.json for identity; code/rr and parent_source; tokenizer snapshots; training_summary.json; predictions/dev_metrics.json and official evaluator log; predictions/*_logits.npy and *_order.json for teacher/selection; submission_receipt.json; observations.zip for a small review bundle.

For an interrupted task attach its whole saved output to the same notebook, keep its ID/configuration and rerun. Auto discovery restores the exact next update. For completed held-out duration extension set TARGET_TOTAL_EPOCHS above the completed count, keeping all other learning/head choices unchanged. The original scheduler horizon is retained. Never resume a different method/architecture's optimizer.

Run 01–05 as separate jobs, then 06 with a completed eligible larger Phase 3 teacher. Run 09 on all completed held-out outputs to compare real metrics; a fallback with identical predictions is not a new improved submission. Optional 07 measures sentence-context value. Optional 08 cold-refits the frozen selected recipe on train+dev; its development score is null. Preserve six distinct method submissions and honest failed results for the report. Check live deadlines/quota before submission; maximum F1 cannot be promised.